In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Next, run this cell to inspect the three datasets. It will identify folder structures, YAML configurations, image counts, XML files, and annotation classes.

In [ ]:
import os
from pathlib import Path
from collections import Counter
import xml.etree.ElementTree as ET
import yaml

# Dataset paths
BASE_DIR = Path("/content/drive/MyDrive/CNN")

DATASETS = {
    "indian": BASE_DIR / "Indian_vehicle_dataset",
    "traffic": BASE_DIR / "traffic_analysis_yolov8_dataset",
    "vehicle": BASE_DIR / "vehicleditect",
}

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

for name, root in DATASETS.items():
    print("\n" + "=" * 65)
    print(f"DATASET: {name}")
    print(f"PATH: {root}")
    print("=" * 65)

    if not root.exists():
        print("ERROR: Dataset folder not found!")
        continue

    # Display top-level contents
    print("\nTop-level contents:")
    for item in sorted(root.iterdir()):
        kind = "DIR " if item.is_dir() else "FILE"
        print(f"  [{kind}] {item.name}")

    # Count images, XML files and YOLO label files
    images = [
        p for p in root.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    ]

    xml_files = list(root.rglob("*.xml"))
    txt_files = [
        p for p in root.rglob("*.txt")
        if p.name.lower() not in {
            "readme.txt", "readme.dataset.txt", "readme.roboflow.txt"
        }
    ]
    yaml_files = list(root.rglob("*.yaml")) + list(root.rglob("*.yml"))

    print(f"\nTotal images: {len(images)}")
    print(f"XML annotation files: {len(xml_files)}")
    print(f"TXT files: {len(txt_files)}")
    print(f"YAML files: {len(yaml_files)}")

    # Display split folders
    print("\nPotential split directories:")
    for p in root.rglob("*"):
        if p.is_dir() and p.name.lower() in {
            "train", "valid", "val", "test"
        }:
            print(f"  {p.relative_to(root)}")

    # Inspect YAML class definitions
    for yp in yaml_files:
        try:
            with yp.open("r", encoding="utf-8") as f:
                config = yaml.safe_load(f)

            if isinstance(config, dict) and "names" in config:
                print(f"\nClasses in {yp.relative_to(root)}:")
                print(config["names"])
        except Exception as e:
            print(f"Could not read {yp.name}: {e}")

    # Inspect XML class names
    if xml_files:
        xml_classes = Counter()

        for xp in xml_files:
            try:
                tree = ET.parse(xp)
                for obj in tree.findall(".//object"):
                    class_name = obj.findtext("name", default="")
                    xml_classes[class_name.strip()] += 1
            except Exception:
                pass

        print("\nClasses found in XML annotations:")
        for cls, count in xml_classes.most_common():
            print(f"  {cls}: {count}")


DATASET: indian
PATH: /content/drive/MyDrive/CNN/Indian_vehicle_dataset

Top-level contents:
  [FILE] 20210427_05_35_43_000_ovRKxW0QbBVfwN9RW4teROP1YgG3_F_4160_3120.jpg
  [FILE] 20210427_05_35_43_000_ovRKxW0QbBVfwN9RW4teROP1YgG3_F_4160_3120.xml
  [FILE] 20210427_07_32_15_000_T4CaxysrQ7W21Wgl1wZWBCtdRjE3_F_3264_2448.jpg
  [FILE] 20210427_07_32_15_000_T4CaxysrQ7W21Wgl1wZWBCtdRjE3_F_3264_2448.xml
  [FILE] 20210427_22_52_25_000_Db2I2d17GLf53JzlQRSP28hD4Ml1_F_2592_1944.jpg
  [FILE] 20210427_22_52_25_000_Db2I2d17GLf53JzlQRSP28hD4Ml1_F_2592_1944.xml
  [FILE] 20210430_19_00_07_000_M0qQbjq7mhasETXjCQGvnoZxK3a2_T_4160_3120.jpg
  [FILE] 20210430_19_00_07_000_M0qQbjq7mhasETXjCQGvnoZxK3a2_T_4160_3120.xml
  [FILE] 20210504_16_08_07_000_x1Al88rRcbgbIm5Y4D4pjFVpufq2_F_4000_3000.jpg
  [FILE] 20210504_16_08_07_000_x1Al88rRcbgbIm5Y4D4pjFVpufq2_F_4000_3000.xml
  [FILE] 20210505_12_39_03_000_x1Al88rRcbgbIm5Y4D4pjFVpufq2_F_4000_3000.jpg
  [FILE] 20210505_12_39_03_000_x1Al88rRcbgbIm5Y4D4pjFVpufq2_F_4000_300

Step 3 — Run the following conversion and YAML preparation cell
This cell creates a new working dataset, converts XML annotations to YOLO format, and writes data.yaml. It does not merge the two existing YOLO datasets yet.

For now, it maps two_wheelers to bike, tempo to auto, and truck_tanker to truck. Review these mappings before running if those classes have different meanings in your dataset.

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET
import random
import shutil
import yaml

# Google Drive paths
BASE = Path("/content/drive/MyDrive/CNN")

XML_SOURCE = BASE / "Indian_vehicle_dataset"
OUTPUT = BASE / "combined_vehicle_dataset"

# Unified class list
CLASSES = [
    "bicycle",
    "bus",
    "car",
    "bike",
    "auto",
    "truck",
    "van",
    "person",
    "concrete_mixer",
    "tractor",
    "vehicle_truck",
]

CLASS_TO_ID = {name: i for i, name in enumerate(CLASSES)}

# Explicit source-label mappings
LABEL_MAP = {
    "bicycle": "bicycle",
    "bus": "bus",
    "car": "car",
    "bike": "bike",
    "motorbike": "bike",
    "two_wheelers": "bike",
    "rickshaw": "auto",
    "auto": "auto",
    "tempo": "auto",
    "truck": "truck",
    "truck_tanker": "truck",
    "van": "van",
    "person": "person",
    "concrete_mixer": "concrete_mixer",
    "tractor": "tractor",
    "vehicle_truck": "vehicle_truck",
}

# Create destination folders
for split in ("train", "valid", "test"):
    (OUTPUT / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUTPUT / "labels" / split).mkdir(parents=True, exist_ok=True)

# Find image/XML pairs
xml_files = sorted(XML_SOURCE.rglob("*.xml"))
pairs = []

for xml_path in xml_files:
    root = ET.parse(xml_path).getroot()
    filename = root.findtext("filename", default=xml_path.stem).strip()

    image_path = xml_path.parent / filename

    if not image_path.exists():
        # Fall back to matching the XML stem
        matches = [
            p for p in xml_path.parent.iterdir()
            if p.is_file()
            and p.stem == xml_path.stem
            and p.suffix.lower() in {".jpg", ".jpeg", ".png"}
        ]
        if not matches:
            print("Missing image:", filename)
            continue
        image_path = matches[0]

    pairs.append((image_path, xml_path))

# Reproducible 80/10/10 split for the XML dataset
random.Random(42).shuffle(pairs)

n = len(pairs)
n_train = int(n * 0.8)
n_valid = int(n * 0.1)

splits = {
    "train": pairs[:n_train],
    "valid": pairs[n_train:n_train + n_valid],
    "test": pairs[n_train + n_valid:],
}

unknown_labels = set()
converted_objects = 0
skipped_objects = 0

for split, split_pairs in splits.items():
    for image_path, xml_path in split_pairs:
        root = ET.parse(xml_path).getroot()
        size = root.find("size")

        if size is None:
            print("Missing image dimensions:", xml_path.name)
            continue

        width = float(size.findtext("width", "0"))
        height = float(size.findtext("height", "0"))

        if width <= 0 or height <= 0:
            print("Invalid dimensions:", xml_path.name)
            continue

        rows = []

        for obj in root.findall("object"):
            original_name = obj.findtext("name", "").strip().lower()

            # Exclude Cart as requested
            if original_name == "cart":
                continue

            mapped_name = LABEL_MAP.get(original_name)

            if mapped_name is None:
                unknown_labels.add(original_name)
                skipped_objects += 1
                continue

            box = obj.find("bndbox")
            if box is None:
                skipped_objects += 1
                continue

            try:
                xmin = float(box.findtext("xmin"))
                ymin = float(box.findtext("ymin"))
                xmax = float(box.findtext("xmax"))
                ymax = float(box.findtext("ymax"))
            except (TypeError, ValueError):
                skipped_objects += 1
                continue

            # Clip boxes to image boundaries
            xmin = max(0, min(xmin, width))
            xmax = max(0, min(xmax, width))
            ymin = max(0, min(ymin, height))
            ymax = max(0, min(ymax, height))

            if xmax <= xmin or ymax <= ymin:
                skipped_objects += 1
                continue

            x_center = ((xmin + xmax) / 2) / width
            y_center = ((ymin + ymax) / 2) / height
            box_width = (xmax - xmin) / width
            box_height = (ymax - ymin) / height

            class_id = CLASS_TO_ID[mapped_name]

            rows.append(
                f"{class_id} {x_center:.6f} {y_center:.6f} "
                f"{box_width:.6f} {box_height:.6f}"
            )

            converted_objects += 1

        # Preserve unique filenames by adding a dataset prefix
        output_name = f"indian_{image_path.name}"
        destination_image = OUTPUT / "images" / split / output_name
        destination_label = OUTPUT / "labels" / split / (
            Path(output_name).stem + ".txt"
        )

        shutil.copy2(image_path, destination_image)
        destination_label.write_text(
            "\n".join(rows) + ("\n" if rows else ""),
            encoding="utf-8"
        )

# Write dataset configuration
config = {
    "path": str(OUTPUT),
    "train": "images/train",
    "val": "images/valid",
    "test": "images/test",
    "nc": len(CLASSES),
    "names": {i: name for i, name in enumerate(CLASSES)},
}

with open(OUTPUT / "data.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, sort_keys=False)

print("\nXML conversion finished.")
print("Matched image/XML pairs:", len(pairs))
print("Converted objects:", converted_objects)
print("Skipped objects:", skipped_objects)
print("Train:", len(splits["train"]))
print("Valid:", len(splits["valid"]))
print("Test:", len(splits["test"]))
print("Unknown labels:", sorted(unknown_labels))
print("Output:", OUTPUT)
print("YAML:", OUTPUT / "data.yaml")

Important: Zero skipped objects and zero unknown classes are good signs, but we still need to verify bounding boxes and class mappings.

Step 2: Inspect the existing YOLO datasets
Before merging, we need to examine the existing labels and their class IDs. Run this cell in Google Colab.

In [ ]:
from pathlib import Path
import yaml
from collections import Counter

BASE = Path("/content/drive/MyDrive/CNN")

DATASETS = {
    "traffic": BASE / "traffic_analysis_yolov8_dataset",
    "vehicle": BASE / "vehicleditect",
}

for dataset_name, root in DATASETS.items():
    print("\n" + "=" * 60)
    print("DATASET:", dataset_name)
    print("=" * 60)

    with open(root / "data.yaml", "r", encoding="utf-8") as f:
        config = yaml.safe_load(f)

    names = config["names"]

    if isinstance(names, list):
        class_names = {i: name for i, name in enumerate(names)}
    else:
        class_names = {int(i): name for i, name in names.items()}

    print("Original classes:")
    print(class_names)

    for split in ["train", "valid", "test"]:
        split_dir = root / split

        if not split_dir.exists():
            print(f"{split}: folder not found")
            continue

        images_dir = split_dir / "images"
        labels_dir = split_dir / "labels"

        images = (
            [p for p in images_dir.rglob("*")
             if p.is_file() and p.suffix.lower() in
             {".jpg", ".jpeg", ".png", ".bmp", ".webp"}]
            if images_dir.exists() else []
        )

        labels = list(labels_dir.rglob("*.txt")) if labels_dir.exists() else []

        class_counts = Counter()
        invalid_lines = 0

        for label_file in labels:
            for line in label_file.read_text(
                encoding="utf-8", errors="ignore"
            ).splitlines():
                parts = line.split()

                if len(parts) != 5:
                    invalid_lines += 1
                    continue

                try:
                    class_id = int(float(parts[0]))
                    coords = [float(v) for v in parts[1:]]

                    if class_id not in class_names:
                        invalid_lines += 1
                        continue

                    if not all(0 <= v <= 1 for v in coords):
                        invalid_lines += 1
                        continue

                    class_counts[class_names[class_id]] += 1

                except ValueError:
                    invalid_lines += 1

        print(f"\n{split}:")
        print("  Images:", len(images))
        print("  Label files:", len(labels))
        print("  Invalid annotation lines:", invalid_lines)
        print("  Object counts:", dict(class_counts))

The output reveals an important issue: the traffic dataset’s training labels were not found in the expected folder, and its validation/test labels appear to have a format or class-ID problem. We should fix this before merging or training YOLO11n.
What the inspection tells us
Dataset / split	Images	Label files	Status
Traffic — train	963	0	Labels may be stored elsewhere or the folder structure differs
Traffic — valid	28	28	825 invalid annotation lines
Traffic — test	6	6	128 invalid annotation lines
Vehicle — train/valid/test	Not included in output	Not included	Need the remaining output


The invalid lines could be caused by label files containing something other than standard YOLO detection annotations (class_id x_center y_center width height), incorrect class IDs, or coordinates outside the expected range. We need to inspect the actual files rather than guess.

Step 1 — Find the traffic labels
Run this cell in Google Colab:

In [ ]:
from pathlib import Path

root = Path("/content/drive/MyDrive/CNN/traffic_analysis_yolov8_dataset")

print("Top-level contents:")
for p in sorted(root.iterdir()):
    print("DIR " if p.is_dir() else "FILE", p.name)

print("\nAll directories containing 'label' or 'image':")
for p in root.rglob("*"):
    if p.is_dir() and ("label" in p.name.lower() or "image" in p.name.lower()):
        print(p)

print("\nFirst 10 label TXT files anywhere in the dataset:")
txt_files = list(root.rglob("*.txt"))
print("Total TXT files:", len(txt_files))

for p in txt_files[:10]:
    print(f"\nFile: {p}")
    try:
        print(p.read_text(encoding="utf-8", errors="replace")[:500])
    except Exception as e:
        print("Could not read:", e)

Step 2 — Inspect the invalid validation/test annotations
Run this second cell to see examples of the lines that failed validation:

In [ ]:
from pathlib import Path

root = Path("/content/drive/MyDrive/CNN/traffic_analysis_yolov8_dataset")

for split in ["valid", "test"]:
    labels_dir = root / split / "labels"

    print(f"\n{'=' * 20} {split.upper()} {'=' * 20}")

    if not labels_dir.exists():
        print("Expected labels folder not found:", labels_dir)
        continue

    shown = 0

    for label_file in sorted(labels_dir.rglob("*.txt")):
        for line_no, line in enumerate(
            label_file.read_text(
                encoding="utf-8", errors="replace"
            ).splitlines(),
            start=1
        ):
            parts = line.split()

            valid = False
            reason = ""

            if len(parts) != 5:
                reason = f"Expected 5 values, found {len(parts)}"
            else:
                try:
                    class_id = int(float(parts[0]))
                    coords = [float(v) for v in parts[1:]]

                    if class_id not in range(9):
                        reason = f"Unexpected class ID: {class_id}"
                    elif not all(0 <= v <= 1 for v in coords):
                        reason = f"Coordinates outside [0, 1]: {coords}"
                    else:
                        valid = True

                except ValueError:
                    reason = "Non-numeric annotation value"

            if not valid:
                print(f"{label_file.name}, line {line_no}: {line!r}")
                print("  Reason:", reason)
                shown += 1

                if shown >= 10:
                    break

        if shown >= 10:
            break

    if shown == 0:
        print("No invalid lines found.")

This confirms that the validation and test annotations use polygon coordinates. We can convert them to bounding boxes for YOLO11n detection. However, we still need to locate the missing training labels before merging.
Next: Run this diagnostic cell in Google Colab
It checks the training folder, counts images and labels in all three datasets, and reports the traffic dataset's class names.

In [ ]:
from pathlib import Path
import yaml

BASE = Path("/content/drive/MyDrive/CNN")

DATASETS = {
    "traffic": BASE / "traffic_analysis_yolov8_dataset",
    "vehicle": BASE / "vehicleditect",
    "combined_xml": BASE / "combined_vehicle_dataset",
}

for name, root in DATASETS.items():
    print("\n" + "=" * 60)
    print("DATASET:", name)
    print("=" * 60)

    yaml_path = root / "data.yaml"

    if yaml_path.exists():
        with open(yaml_path, "r", encoding="utf-8") as f:
            config = yaml.safe_load(f)
        print("Classes:", config.get("names"))
    else:
        print("data.yaml not found")

    for split in ["train", "valid", "test"]:
        split_dir = root / split

        if not split_dir.exists():
            continue

        images_dir = split_dir / "images"
        labels_dir = split_dir / "labels"

        image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

        images = (
            [p for p in images_dir.rglob("*")
             if p.is_file() and p.suffix.lower() in image_exts]
            if images_dir.exists() else []
        )

        labels = (
            list(labels_dir.rglob("*.txt"))
            if labels_dir.exists() else []
        )

        print(
            f"{split}: images={len(images)}, "
            f"label_files={len(labels)}"
        )

    # Show any labels in the traffic training directory
    if name == "traffic":
        print("\nTraining directory contents:")
        train_dir = root / "train"

        for p in list(train_dir.rglob("*"))[:25]:
            print(("DIR " if p.is_dir() else "FILE"), p)

Good, we have the vehicle dataset counts and confirmed that your XML dataset already has the unified 11-class mapping.

There are two things to address before merging:
- Vehicle dataset training: 952 images but 7,556 label files. That is unusual and may indicate nested directories, duplicate labels, or a folder-structure issue. We should check how the images and labels match.
- Traffic dataset: its validation/test annotations use polygon coordinates, and the training labels were not found in the expected location.

Step 1 — Check image-label matching
Run this diagnostic cell in Colab. It checks whether each image has a matching .txt label file with the same filename stem.

In [ ]:
from pathlib import Path
from collections import Counter

BASE = Path("/content/drive/MyDrive/CNN")

DATASETS = {
    "traffic": BASE / "traffic_analysis_yolov8_dataset",
    "vehicle": BASE / "vehicleditect",
    "combined_xml": BASE / "combined_vehicle_dataset",
}

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

for name, root in DATASETS.items():
    print("\n" + "=" * 65)
    print("DATASET:", name)
    print("=" * 65)

    for split in ["train", "valid", "test"]:
        split_dir = root / split
        if not split_dir.exists():
            continue

        images_dir = split_dir / "images"
        labels_dir = split_dir / "labels"

        images = (
            [p for p in images_dir.rglob("*")
             if p.is_file() and p.suffix.lower() in IMAGE_EXTS]
            if images_dir.exists() else []
        )
        labels = (
            [p for p in labels_dir.rglob("*.txt") if p.is_file()]
            if labels_dir.exists() else []
        )

        # Compare relative paths without extensions, preserving subfolders.
        image_keys = {
            p.relative_to(images_dir).with_suffix("").as_posix()
            for p in images
        } if images_dir.exists() else set()

        label_keys = {
            p.relative_to(labels_dir).with_suffix("").as_posix()
            for p in labels
        } if labels_dir.exists() else set()

        matched = image_keys & label_keys
        images_without_labels = image_keys - label_keys
        labels_without_images = label_keys - image_keys

        print(f"\n{split}:")
        print("  Images:", len(images))
        print("  Label files:", len(labels))
        print("  Matching image-label pairs:", len(matched))
        print("  Images without labels:", len(images_without_labels))
        print("  Labels without images:", len(labels_without_images))

        if images_without_labels:
            print("  Example images without labels:",
                  list(sorted(images_without_labels))[:5])

        if labels_without_images:
            print("  Example labels without images:",
                  list(sorted(labels_without_images))[:5])

We’ve identified two important dataset problems:
1. Traffic dataset: all 963 training images lack matching label files. We should not use them for supervised detection training until we locate their annotations or confirm they are intentionally unlabeled.
2. Vehicle dataset: 951 of 952 training images have matching labels, but there are 6,605 extra .txt files. These may be leftover annotations or files in a different directory structure. We should not copy those extra files into the merged dataset.
The traffic validation and test sets do have matching annotation files. Their polygon annotations can be converted to bounding boxes.


Next step: inspect the folder structures
Run this single Colab cell. It checks whether the vehicle dataset’s extra labels are in nested folders and searches for any traffic training labels elsewhere under your CNN directory.

In [ ]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/CNN")

# 1. Inspect the vehicle training directory structure
vehicle_train = BASE / "vehicleditect" / "train"

print("=" * 60)
print("VEHICLE TRAIN DIRECTORY")
print("=" * 60)

for p in list(vehicle_train.iterdir()):
    if p.is_dir():
        files = [x for x in p.rglob("*") if x.is_file()]
        print(f"DIR  {p.name}: {len(files)} files")
    else:
        print("FILE", p.name)

# Count annotation files by parent directory
labels_dir = vehicle_train / "labels"

if labels_dir.exists():
    print("\nVehicle label subdirectories:")
    for p in sorted(labels_dir.rglob("*")):
        if p.is_dir():
            txt_count = sum(1 for x in p.glob("*.txt") if x.is_file())
            if txt_count:
                print(f"{p}: {txt_count} TXT files")

# 2. Search for labels corresponding to traffic training images
traffic = BASE / "traffic_analysis_yolov8_dataset"
traffic_train_images = traffic / "train" / "images"

image_stems = {
    p.stem
    for p in traffic_train_images.rglob("*")
    if p.is_file() and p.suffix.lower() in
    {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
}

print("\n" + "=" * 60)
print("SEARCH FOR TRAFFIC TRAINING LABELS")
print("=" * 60)
print("Traffic training images:", len(image_stems))

found = []

for txt in BASE.rglob("*.txt"):
    # Ignore files already in the traffic validation/test folders
    if traffic in txt.parents:
        continue

    if txt.stem in image_stems:
        found.append(txt)

print("Matching TXT files found elsewhere:", len(found))

for p in found[:20]:
    print(p)

# 3. Summarize the combined XML dataset
combined = BASE / "combined_vehicle_dataset"

print("\n" + "=" * 60)
print("COMBINED XML DATASET")
print("=" * 60)

for split in ["train", "valid", "test"]:
    for kind in ["images", "labels"]:
        folder = combined / split / kind
        if folder.exists():
            count = sum(1 for p in folder.rglob("*") if p.is_file())
            print(f"{split}/{kind}: {count} files")

Now we know the traffic training labels are not present elsewhere under your CNN directory. The vehicle training folder also has many extra label files, so we should only use labels that match actual images.

My recommendation: build a new merged dataset in a separate folder and leave your original datasets untouched. Before merging, let's check the vehicle labels' contents and confirm whether the extra files are usable.


Step 1 — Inspect the vehicle training labels
Run this cell in Google Colab:

In [ ]:
from pathlib import Path
from collections import Counter
import yaml

BASE = Path("/content/drive/MyDrive/CNN")
ROOT = BASE / "vehicleditect"

with open(ROOT / "data.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

names = config["names"]
if isinstance(names, list):
    names = {i: name for i, name in enumerate(names)}
else:
    names = {int(k): v for k, v in names.items()}

for split in ["train", "valid", "test"]:
    images_dir = ROOT / split / "images"
    labels_dir = ROOT / split / "labels"

    image_stems = {
        p.stem for p in images_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in
        {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    }

    label_files = list(labels_dir.rglob("*.txt"))
    matching_labels = [p for p in label_files if p.stem in image_stems]

    counts = Counter()
    invalid = 0
    examples = []

    for p in matching_labels:
        for line in p.read_text(encoding="utf-8", errors="replace").splitlines():
            parts = line.split()

            if len(parts) != 5:
                invalid += 1
                if len(examples) < 3:
                    examples.append((p.name, line, "Expected 5 values"))
                continue

            try:
                class_id = int(parts[0])
                coords = [float(x) for x in parts[1:]]

                if class_id not in names or not all(0 <= x <= 1 for x in coords):
                    invalid += 1
                    if len(examples) < 3:
                        examples.append((p.name, line, "Invalid class ID or coordinates"))
                    continue

                counts[names[class_id]] += 1

            except ValueError:
                invalid += 1
                if len(examples) < 3:
                    examples.append((p.name, line, "Non-numeric values"))

    print(f"\n{split.upper()}")
    print("Images:", len(image_stems))
    print("Label files:", len(label_files))
    print("Labels matching images:", len(matching_labels))
    print("Invalid annotation lines:", invalid)
    print("Object counts:", dict(counts))

    if examples:
        print("Invalid examples:")
        for example in examples:
            print(" ", example)

Step 1 — Create the merged dataset
Run this code cell in Google Colab. It creates a new output folder so your original datasets and converted XML dataset remain unchanged.

The merged dataset will use the 11 agreed classes and preserve the source train/validation/test splits.

In [ ]:
from pathlib import Path
from collections import Counter
import shutil
import random
import yaml

# ============================================================
# CONFIGURATION
# ============================================================

BASE = Path("/content/drive/MyDrive/CNN")

VEHICLE = BASE / "vehicleditect"
TRAFFIC = BASE / "traffic_analysis_yolov8_dataset"
XML_DATA = BASE / "combined_vehicle_dataset"

# New folder: original datasets are not modified.
OUTPUT = BASE / "merged_vehicle_dataset_fixed"

CLASS_NAMES = [
    "bicycle",        # 0
    "bus",            # 1
    "car",            # 2
    "bike",           # 3
    "auto",           # 4
    "truck",          # 5
    "van",            # 6
    "person",         # 7
    "concrete_mixer", # 8
    "tractor",        # 9
    "vehicle_truck",  # 10
]

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Set True only if you intentionally want to replace this output folder.
OVERWRITE_OUTPUT = False

if OUTPUT.exists():
    if not OVERWRITE_OUTPUT:
        raise FileExistsError(
            f"{OUTPUT} already exists. Set OVERWRITE_OUTPUT=True "
            "only if you want to replace this generated output."
        )
    shutil.rmtree(OUTPUT)

for split in ["train", "valid", "test"]:
    (OUTPUT / split / "images").mkdir(parents=True, exist_ok=True)
    (OUTPUT / split / "labels").mkdir(parents=True, exist_ok=True)

# ============================================================
# HELPERS
# ============================================================

def load_class_names(yaml_path):
    with open(yaml_path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)

    names = cfg["names"]
    if isinstance(names, list):
        return {i: str(name) for i, name in enumerate(names)}
    return {int(i): str(name) for i, name in names.items()}


def find_images(folder):
    if not folder.exists():
        return []
    return sorted(
        p for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    )


def copy_pair(image_path, label_lines, split, prefix, stats):
    # Include a relative-path-derived identifier to avoid basename collisions.
    unique_name = f"{prefix}_{image_path.stem}"
    image_dest = OUTPUT / split / "images" / f"{unique_name}{image_path.suffix.lower()}"
    label_dest = OUTPUT / split / "labels" / f"{unique_name}.txt"

    if image_dest.exists() or label_dest.exists():
        raise RuntimeError(f"Filename collision detected: {unique_name}")

    shutil.copy2(image_path, image_dest)
    label_dest.write_text(
        "\n".join(label_lines) + ("\n" if label_lines else ""),
        encoding="utf-8"
    )

    stats[split]["images"] += 1
    stats[split]["objects"] += len(label_lines)


def read_yolo_detection_label(label_path, source_names, stats):
    """Read standard YOLO detection boxes and remap class IDs by name."""
    mapped_lines = []

    if not label_path.exists():
        return None

    for line_no, line in enumerate(
        label_path.read_text(encoding="utf-8", errors="replace").splitlines(),
        start=1
    ):
        parts = line.split()
        if not parts:
            continue

        if len(parts) != 5:
            stats["invalid_lines"] += 1
            continue

        try:
            old_id = int(parts[0])
            coords = [float(v) for v in parts[1:]]
        except ValueError:
            stats["invalid_lines"] += 1
            continue

        if old_id not in source_names:
            stats["invalid_lines"] += 1
            continue

        if not all(0 <= v <= 1 for v in coords):
            stats["invalid_lines"] += 1
            continue

        source_class = source_names[old_id].strip().lower()

        # Remap by source class name, not the old numeric ID.
        mapping = {
            "bicycle": "bicycle",
            "bus": "bus",
            "car": "car",
            "bike": "bike",
            "motorbike": "bike",
            "rickshaw": "auto",
            "auto": "auto",
            "truck": "truck",
            "van": "van",
            "person": "person",
            "concrete_mixer": "concrete_mixer",
            "tractor": "tractor",
            "vehicle_truck": "vehicle_truck",
            "cart": None,
            "suzuki": None,  # unresolved; exclude rather than mislabel
        }

        target_name = mapping.get(source_class, None)

        if target_name is None:
            stats["excluded_objects"][source_class] += 1
            continue

        target_id = CLASS_NAMES.index(target_name)
        mapped_lines.append(
            str(target_id) + " " + " ".join(f"{v:.8f}" for v in coords)
        )

    return mapped_lines


def polygon_to_bbox_line(line, source_names, stats):
    """
    Convert a polygon annotation:
        class_id x1 y1 x2 y2 ...
    into a YOLO detection box:
        class_id xc yc width height
    """
    parts = line.split()
    if len(parts) < 7 or (len(parts) - 1) % 2 != 0:
        stats["invalid_lines"] += 1
        return None

    try:
        old_id = int(parts[0])
        values = [float(v) for v in parts[1:]]
    except ValueError:
        stats["invalid_lines"] += 1
        return None

    if old_id not in source_names:
        stats["invalid_lines"] += 1
        return None

    if not all(0 <= v <= 1 for v in values):
        stats["invalid_lines"] += 1
        return None

    source_class = source_names[old_id].strip().lower()

    mapping = {
        "bike": "bike",
        "bus": "bus",
        "car": "car",
        "cart": None,
        "person": "person",
        "rickshaw": "auto",
        "suzuki": None,  # unresolved; excluded for now
        "truck": "truck",
        "van": "van",
    }

    target_name = mapping.get(source_class, None)

    if target_name is None:
        stats["excluded_objects"][source_class] += 1
        return None

    xs = values[0::2]
    ys = values[1::2]

    xmin, xmax = min(xs), max(xs)
    ymin, ymax = min(ys), max(ys)

    width = xmax - xmin
    height = ymax - ymin

    if width <= 0 or height <= 0:
        stats["invalid_lines"] += 1
        return None

    xc = (xmin + xmax) / 2
    yc = (ymin + ymax) / 2

    target_id = CLASS_NAMES.index(target_name)

    return (
        f"{target_id} {xc:.8f} {yc:.8f} "
        f"{width:.8f} {height:.8f}"
    )


# ============================================================
# MERGE SOURCES
# ============================================================

stats = {
    split: {"images": 0, "objects": 0}
    for split in ["train", "valid", "test"]
}

stats["invalid_lines"] = 0
stats["excluded_objects"] = Counter()

# ------------------------------------------------------------
# A. VEHICLE DATASET
# ------------------------------------------------------------

vehicle_names = load_class_names(VEHICLE / "data.yaml")

for split in ["train", "valid", "test"]:
    images_dir = VEHICLE / split / "images"
    labels_dir = VEHICLE / split / "labels"

    for image_path in find_images(images_dir):
        label_path = labels_dir / f"{image_path.stem}.txt"

        # Skip images with no matching annotation.
        if not label_path.exists():
            print("Skipping image without label:", image_path.name)
            continue

        lines = read_yolo_detection_label(
            label_path, vehicle_names, stats
        )

        if lines is None:
            continue

        copy_pair(
            image_path,
            lines,
            split,
            "vehicle",
            stats
        )

# ------------------------------------------------------------
# B. XML-CONVERTED DATASET
# Images and labels are stored under images/<split>
# and labels/<split>, respectively.
# ------------------------------------------------------------

xml_names = load_class_names(XML_DATA / "data.yaml")

for split in ["train", "valid", "test"]:
    images_dir = XML_DATA / "images" / split
    labels_dir = XML_DATA / "labels" / split

    for image_path in find_images(images_dir):
        label_path = labels_dir / f"{image_path.stem}.txt"

        if not label_path.exists():
            print("Skipping XML image without label:", image_path.name)
            continue

        lines = []

        for line in label_path.read_text(
            encoding="utf-8", errors="replace"
        ).splitlines():
            parts = line.split()

            if len(parts) != 5:
                stats["invalid_lines"] += 1
                continue

            try:
                class_id = int(parts[0])
                coords = [float(v) for v in parts[1:]]
            except ValueError:
                stats["invalid_lines"] += 1
                continue

            if not (0 <= class_id < len(CLASS_NAMES)):
                stats["invalid_lines"] += 1
                continue

            if not all(0 <= v <= 1 for v in coords):
                stats["invalid_lines"] += 1
                continue

            lines.append(
                str(class_id) + " " + " ".join(f"{v:.8f}" for v in coords)
            )

        copy_pair(image_path, lines, split, "xml", stats)

# ------------------------------------------------------------
# C. TRAFFIC VALIDATION + TEST ONLY
# Training labels are missing, so traffic train is excluded.
# Polygon labels are converted to detection boxes.
# ------------------------------------------------------------

traffic_names = load_class_names(TRAFFIC / "data.yaml")

for split in ["valid", "test"]:
    images_dir = TRAFFIC / split / "images"
    labels_dir = TRAFFIC / split / "labels"

    for image_path in find_images(images_dir):
        label_path = labels_dir / f"{image_path.stem}.txt"

        if not label_path.exists():
            print("Skipping traffic image without label:", image_path.name)
            continue

        lines = []
        for line in label_path.read_text(
            encoding="utf-8", errors="replace"
        ).splitlines():
            if not line.strip():
                continue

            converted = polygon_to_bbox_line(
                line, traffic_names, stats
            )

            if converted is not None:
                lines.append(converted)

        copy_pair(
            image_path,
            lines,
            split,
            "traffic",
            stats
        )

# ------------------------------------------------------------
# WRITE YAML
# ------------------------------------------------------------

yaml_config = {
    "path": str(OUTPUT),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "names": {i: name for i, name in enumerate(CLASS_NAMES)},
    "nc": len(CLASS_NAMES),
}

with open(OUTPUT / "data.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(yaml_config, f, sort_keys=False, allow_unicode=True)

print("\nMERGE COMPLETE")
print("Output:", OUTPUT)
print("\nSplit totals:")

for split in ["train", "valid", "test"]:
    print(
        f"{split}: {stats[split]['images']} images, "
        f"{stats[split]['objects']} objects"
    )

print("\nInvalid annotation lines:", stats["invalid_lines"])
print("Excluded object counts:", dict(stats["excluded_objects"]))
print("YAML:", OUTPUT / "data.yaml")

In [ ]:
from pathlib import Path
from collections import Counter
import yaml

ROOT = Path("/content/drive/MyDrive/CNN/merged_vehicle_dataset_fixed")

with open(ROOT / "data.yaml", "r") as f:
    data = yaml.safe_load(f)

names = data["names"]
if isinstance(names, list):
    names = dict(enumerate(names))
else:
    names = {int(k): v for k, v in names.items()}

counts = Counter()

for split in ["train", "valid", "test"]:
    for label_file in (ROOT / split / "labels").glob("*.txt"):
        for line in label_file.read_text().splitlines():
            parts = line.split()
            if parts:
                try:
                    counts[int(parts[0])] += 1
                except ValueError:
                    pass

print("Merged dataset class counts:")
for class_id, name in sorted(names.items()):
    print(f"{class_id}: {name} -> {counts[class_id]}")

print("\nTarget class verification:")
expected = {
    "concrete_mixer": 11,
    "tractor": 22,
    "vehicle_truck": 35,
}

for name, expected_count in expected.items():
    class_id = next(i for i, n in names.items() if n == name)
    actual = counts[class_id]
    status = "PASS" if actual >= expected_count else "CHECK"
    print(f"{name}: expected at least {expected_count}, found {actual} [{status}]")


split is not good so we will do 70_20_10

Let's create a clean 70% train / 20% validation / 10% test split for your vehicle-detection dataset.

We'll keep your existing dataset unchanged and create a new folder. I'll give you the Colab code step by step.


Step 1: Check your dataset structure
Run this cell in Google Colab:

In [ ]:
from pathlib import Path

DATASET = Path(
    "/content/drive/MyDrive/CNN/merged_vehicle_dataset"
)

for path in sorted(DATASET.rglob("*")):
    if path.is_dir():
        print("📁", path.relative_to(DATASET))

print("\nDataset YAML exists:", (DATASET / "data.yaml").exists())

# Create the new dataset split
This code will:
- Combine images from your existing train, valid, and test folders.
- Match each image with its label file.
- Check annotation formats and class IDs.
- Split the dataset into 70% train, 20% validation, and 10% test.
- Save the new dataset separately, leaving your original dataset unchanged.

In [ ]:
# from pathlib import Path
# from collections import Counter
# import random
# import shutil
# import yaml

# # -----------------------------
# # 1. Configuration
# # -----------------------------
# SOURCE = Path(
#     "/content/drive/MyDrive/CNN/merged_vehicle_dataset"
# )

# OUTPUT = Path(
#     "/content/drive/MyDrive/CNN/vehicle_dataset_70_20_10"
# )

# SEED = 42
# random.seed(SEED)

# IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# assert (SOURCE / "data.yaml").exists(), "data.yaml not found!"
# assert not OUTPUT.exists(), (
#     f"Output already exists: {OUTPUT}\n"
#     "Choose a new output folder or inspect the existing folder first."
# )

# # -----------------------------
# # 2. Read class configuration
# # -----------------------------
# with open(SOURCE / "data.yaml", "r") as f:
#     config = yaml.safe_load(f)

# names = config["names"]

# if isinstance(names, list):
#     names = {i: name for i, name in enumerate(names)}
# else:
#     names = {int(k): v for k, v in names.items()}

# print("Classes:", names)

# # -----------------------------
# # 3. Find image-label pairs
# # -----------------------------
# records = []
# missing_labels = []
# invalid_labels = []

# for split in ["train", "valid", "test"]:
#     image_dir = SOURCE / split / "images"
#     label_dir = SOURCE / split / "labels"

#     if not image_dir.exists():
#         print(f"Skipping missing image folder: {image_dir}")
#         continue

#     for image_path in image_dir.rglob("*"):
#         if (
#             not image_path.is_file()
#             or image_path.suffix.lower() not in IMAGE_EXTENSIONS
#         ):
#             continue

#         label_path = label_dir / f"{image_path.stem}.txt"

#         if not label_path.exists():
#             missing_labels.append(str(image_path))
#             continue

#         classes_in_image = set()
#         valid_lines = 0
#         errors = []

#         with open(label_path, "r") as f:
#             lines = f.readlines()

#         for line_no, line in enumerate(lines, start=1):
#             parts = line.strip().split()

#             if not parts:
#                 continue

#             try:
#                 if len(parts) != 5:
#                     raise ValueError(
#                         f"Expected 5 values, found {len(parts)}"
#                     )

#                 class_id = int(parts[0])
#                 coords = [float(v) for v in parts[1:]]

#                 if class_id not in names:
#                     raise ValueError(f"Unknown class ID {class_id}")

#                 if not all(0 <= v <= 1 for v in coords):
#                     raise ValueError("Coordinates outside [0, 1]")

#                 if coords[2] <= 0 or coords[3] <= 0:
#                     raise ValueError("Non-positive box width/height")

#                 classes_in_image.add(class_id)
#                 valid_lines += 1

#             except Exception as e:
#                 errors.append(f"Line {line_no}: {e}")

#         if errors:
#             invalid_labels.append({
#                 "image": str(image_path),
#                 "label": str(label_path),
#                 "errors": errors
#             })
#             continue

#         records.append({
#             "image": image_path,
#             "label": label_path,
#             "classes": classes_in_image,
#             "objects": valid_lines
#         })

# print("\nSource audit")
# print("Valid image-label pairs:", len(records))
# print("Images missing labels:", len(missing_labels))
# print("Labels with invalid lines:", len(invalid_labels))

# # Save an audit report before stopping if there are problems.
# if missing_labels or invalid_labels:
#     print("\nPlease review the problems before splitting.")
#     print("First missing-label examples:", missing_labels[:5])
#     print("First invalid-label examples:", invalid_labels[:5])

#     raise ValueError(
#         "Dataset audit found missing or invalid annotations. "
#         "Fix or explicitly resolve these before creating the new split."
#     )

# assert len(records) > 0, "No valid image-label pairs found!"

# # -----------------------------
# # 4. Shuffle and split
# # -----------------------------
# random.shuffle(records)

# n = len(records)
# n_train = round(n * 0.70)
# n_valid = round(n * 0.20)

# train_records = records[:n_train]
# valid_records = records[n_train:n_train + n_valid]
# test_records = records[n_train + n_valid:]

# splits = {
#     "train": train_records,
#     "valid": valid_records,
#     "test": test_records
# }

# # -----------------------------
# # 5. Copy paired images/labels
# # -----------------------------
# for split, split_records in splits.items():
#     image_out = OUTPUT / split / "images"
#     label_out = OUTPUT / split / "labels"

#     image_out.mkdir(parents=True, exist_ok=True)
#     label_out.mkdir(parents=True, exist_ok=True)

#     for i, record in enumerate(split_records):
#         # Unique filenames avoid collisions between source folders.
#         filename = f"{split}_{i:06d}{record['image'].suffix.lower()}"

#         shutil.copy2(record["image"], image_out / filename)
#         shutil.copy2(
#             record["label"],
#             label_out / f"{Path(filename).stem}.txt"
#         )

# # -----------------------------
# # 6. Create new data.yaml
# # -----------------------------
# new_config = {
#     "path": str(OUTPUT),
#     "train": "train/images",
#     "val": "valid/images",
#     "test": "test/images",
#     "nc": len(names),
#     "names": [names[i] for i in range(len(names))]
# }

# with open(OUTPUT / "data.yaml", "w") as f:
#     yaml.safe_dump(new_config, f, sort_keys=False)

# # -----------------------------
# # 7. Print results
# # -----------------------------
# print("\nNEW DATASET CREATED")
# print("Location:", OUTPUT)

# for split, split_records in splits.items():
#     objects = sum(r["objects"] for r in split_records)
#     print(
#         f"{split}: {len(split_records)} images, "
#         f"{objects} objects, "
#         f"{len(split_records) / n:.2%}"
#     )

# print("Total images:", n)
# print("Total objects:", sum(r["objects"] for r in records))
# print("YAML:", OUTPUT / "data.yaml")

In [ ]:
!pip install -q iterative-stratification

It creates:
- 70% Train
- 20% Validation
- 10% Test
It uses multilabel stratification because each image can contain multiple vehicle classes.
Stratified 70:20:10 split code

In [ ]:
from pathlib import Path
from collections import Counter
import shutil
import yaml
import numpy as np
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

# Paths
SOURCE = Path("/content/drive/MyDrive/CNN/merged_vehicle_dataset")
OUTPUT = Path("/content/drive/MyDrive/CNN/vehicle_dataset_stratified_70_20_10")

SEED = 42
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

if not SOURCE.exists():
    raise FileNotFoundError(f"Dataset not found: {SOURCE}")

if OUTPUT.exists():
    raise FileExistsError(
        f"Output already exists: {OUTPUT}\n"
        "Choose another output path or remove the old folder yourself."
    )

# Read YAML
with open(SOURCE / "data.yaml", "r") as f:
    config = yaml.safe_load(f)

names = config["names"]
if isinstance(names, list):
    names = {i: name for i, name in enumerate(names)}
else:
    names = {int(k): v for k, v in names.items()}

class_ids = sorted(names)
num_classes = len(class_ids)
class_to_column = {cid: i for i, cid in enumerate(class_ids)}

# Collect image-label pairs and class presence
records = []
missing_labels = []
invalid_labels = []

for split in ["train", "valid", "test"]:
    image_dir = SOURCE / split / "images"
    label_dir = SOURCE / split / "labels"

    if not image_dir.exists():
        continue

    for image_path in image_dir.rglob("*"):
        if not image_path.is_file() or image_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue

        label_path = label_dir / f"{image_path.stem}.txt"

        if not label_path.exists():
            missing_labels.append(str(image_path))
            continue

        present_classes = set()
        object_count = 0
        errors = []

        with open(label_path, "r") as f:
            lines = f.readlines()

        for line_no, line in enumerate(lines, start=1):
            parts = line.split()

            # Empty label files are allowed for genuine background images.
            if not parts:
                continue

            try:
                if len(parts) != 5:
                    raise ValueError(f"Expected 5 values; got {len(parts)}")

                cid = int(parts[0])
                coords = np.array([float(v) for v in parts[1:]], dtype=float)

                if cid not in names:
                    raise ValueError(f"Unknown class ID: {cid}")

                if not np.isfinite(coords).all():
                    raise ValueError("Non-finite coordinates")

                x, y, w, h = coords

                if not (
                    0 <= x <= 1 and 0 <= y <= 1
                    and 0 < w <= 1 and 0 < h <= 1
                    and x - w / 2 >= -1e-6
                    and y - h / 2 >= -1e-6
                    and x + w / 2 <= 1 + 1e-6
                    and y + h / 2 <= 1 + 1e-6
                ):
                    raise ValueError("Invalid bounding-box coordinates")

                present_classes.add(cid)
                object_count += 1

            except Exception as e:
                errors.append(f"Line {line_no}: {e}")

        if errors:
            invalid_labels.append({
                "image": image_path.name,
                "errors": errors
            })
            continue

        records.append({
            "image": image_path,
            "label": label_path,
            "classes": present_classes,
            "objects": object_count
        })

print("Valid image-label pairs:", len(records))
print("Missing labels:", len(missing_labels))
print("Files with invalid annotations:", len(invalid_labels))

if missing_labels or invalid_labels:
    print("Missing label examples:", missing_labels[:5])
    print("Invalid annotation examples:", invalid_labels[:5])
    raise ValueError(
        "Resolve missing or invalid annotations before splitting."
    )

if len(records) < 3:
    raise ValueError("Not enough images to create three splits.")

# Create multilabel matrix
Y = np.zeros((len(records), num_classes), dtype=int)

for row, record in enumerate(records):
    for cid in record["classes"]:
        Y[row, class_to_column[cid]] = 1

for cid, count in zip(class_ids, Y.sum(axis=0)):
    if count < 3:
        print(
            f"Warning: class '{names[cid]}' occurs in only {count} images. "
            "It cannot be guaranteed in every split."
        )

# First split: 70% train, 30% remainder
indices = np.arange(len(records))

split1 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=SEED
)

train_idx, remainder_idx = next(
    split1.split(np.zeros((len(records), 1)), Y)
)

# Second split: 20% validation, 10% test overall
split2 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=1/3,
    random_state=SEED
)

val_local_idx, test_local_idx = next(
    split2.split(
        np.zeros((len(remainder_idx), 1)),
        Y[remainder_idx]
    )
)

val_idx = remainder_idx[val_local_idx]
test_idx = remainder_idx[test_local_idx]

splits = {
    "train": train_idx,
    "valid": val_idx,
    "test": test_idx
}

# Copy images and labels together
for split, split_indices in splits.items():
    image_out = OUTPUT / split / "images"
    label_out = OUTPUT / split / "labels"

    image_out.mkdir(parents=True, exist_ok=True)
    label_out.mkdir(parents=True, exist_ok=True)

    for idx in split_indices:
        record = records[idx]
        filename = f"image_{idx:06d}{record['image'].suffix.lower()}"

        shutil.copy2(record["image"], image_out / filename)
        shutil.copy2(
            record["label"],
            label_out / f"{Path(filename).stem}.txt"
        )

# Create YOLO data.yaml
new_config = {
    "path": str(OUTPUT),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": num_classes,
    "names": [names[cid] for cid in class_ids]
}

with open(OUTPUT / "data.yaml", "w") as f:
    yaml.safe_dump(new_config, f, sort_keys=False)

# Report results
print("\nSTRATIFIED 70:20:10 SPLIT COMPLETE")
print("Output folder:", OUTPUT)

for split, split_indices in splits.items():
    class_image_counts = Counter()
    object_total = 0

    for idx in split_indices:
        record = records[idx]
        object_total += record["objects"]

        for cid in record["classes"]:
            class_image_counts[names[cid]] += 1

    print(
        f"\n{split.upper()}: {len(split_indices)} images "
        f"({len(split_indices)/len(records):.2%}), "
        f"{object_total} objects"
    )
    print("Images containing each class:", dict(class_image_counts))

print("\nTotal images:", len(records))
print("Total objects:", sum(r["objects"] for r in records))
print("YAML:", OUTPUT / "data.yaml")

These warnings mean concrete_mixer, tractor, and vehicle_truck appear in 0 images in your merged dataset. The stratified splitter cannot distribute classes that are not present.

This is not necessarily an error. It likely means these classes were defined in data.yaml but did not make it into merged_vehicle_dataset.
Step 1: Check whether these classes exist in your merged labels
Run this cell in Google Colab:

In [ ]:
from pathlib import Path
from collections import Counter
import yaml

SOURCE = Path("/content/drive/MyDrive/CNN/merged_vehicle_dataset")

with open(SOURCE / "data.yaml", "r") as f:
    data = yaml.safe_load(f)

names = data["names"]
if isinstance(names, list):
    names = dict(enumerate(names))

counts = Counter()

for split in ["train", "valid", "test"]:
    label_dir = SOURCE / split / "labels"

    for label_file in label_dir.glob("*.txt"):
        with open(label_file, "r") as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    counts[int(float(parts[0]))] += 1

print("Class distribution in merged dataset:\n")

for class_id, class_name in sorted(names.items(), key=lambda x: int(x[0])):
    print(f"{class_id}: {class_name:20s} -> {counts[int(class_id)]} objects")

In [ ]:
from pathlib import Path
from collections import Counter
import xml.etree.ElementTree as ET
import yaml

BASE = Path("/content/drive/MyDrive/CNN")

XML_DATASET = BASE / "Indian_vehicle_dataset"
CONVERTED = BASE / "combined_vehicle_dataset"
MERGED = BASE / "merged_vehicle_dataset"

TARGET_CLASSES = {
    "concrete_mixer",
    "tractor",
    "vehicle_truck",
}

# Check original XML annotations
print("=" * 60)
print("1. ORIGINAL XML ANNOTATIONS")
print("=" * 60)

xml_counts = Counter()

for xml_file in XML_DATASET.rglob("*.xml"):
    try:
        root = ET.parse(xml_file).getroot()
        for obj in root.findall("object"):
            name = (obj.findtext("name") or "").strip().lower()
            xml_counts[name] += 1
    except Exception as e:
        print("Could not read:", xml_file.name, e)

for cls in sorted(TARGET_CLASSES):
    print(f"{cls}: {xml_counts[cls]} objects in original XMLs")

# Check YOLO label files
def count_yolo_labels(dataset_path):
    counts = Counter()

    yaml_path = dataset_path / "data.yaml"
    if not yaml_path.exists():
        print(f"Missing YAML: {yaml_path}")
        return counts

    with open(yaml_path, "r") as f:
        config = yaml.safe_load(f)

    names = config["names"]
    if isinstance(names, list):
        names = dict(enumerate(names))

    id_to_name = {int(k): str(v).lower() for k, v in names.items()}

    for label_file in dataset_path.glob("*/labels/*.txt"):
        with open(label_file, "r") as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                try:
                    class_id = int(float(parts[0]))
                    counts[id_to_name.get(class_id, f"unknown_id_{class_id}")] += 1
                except (ValueError, IndexError):
                    continue

    return counts

for title, folder in [
    ("2. CONVERTED XML DATASET", CONVERTED),
    ("3. MERGED DATASET", MERGED),
]:
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)

    counts = count_yolo_labels(folder)
    for cls in sorted(TARGET_CLASSES):
        print(f"{cls}: {counts[cls]} objects")


# Step 1: Check the original XML class names
Run this cell in Colab to see the exact spelling and capitalization of these classes in your XML files.

In [ ]:
from pathlib import Path
from collections import Counter
import xml.etree.ElementTree as ET

XML_DIR = Path(
    "/content/drive/MyDrive/CNN/Indian_vehicle_dataset"
)

counts = Counter()

for xml_file in XML_DIR.rglob("*.xml"):
    try:
        root = ET.parse(xml_file).getroot()

        for obj in root.findall("object"):
            name = (obj.findtext("name") or "").strip()
            if any(x in name.lower() for x in [
                "concrete", "tractor", "vehicle_truck"
            ]):
                counts[name] += 1

    except Exception as e:
        print(f"Error reading {xml_file.name}: {e}")

print("Exact class names found:")
for name, count in counts.items():
    print(f"{name!r}: {count} objects")

# Inspect the converted dataset's class mapping

In [ ]:
from pathlib import Path
import yaml

CONVERTED = Path(
    "/content/drive/MyDrive/CNN/combined_vehicle_dataset"
)

with open(CONVERTED / "data.yaml", "r") as f:
    config = yaml.safe_load(f)

print("Class mapping in converted dataset:")
names = config["names"]

if isinstance(names, dict):
    for class_id, name in names.items():
        print(f"{class_id}: {name}")
else:
    for class_id, name in enumerate(names):
        print(f"{class_id}: {name}")

print("\nLabel folders:")
for split in ["train", "valid", "test"]:
    folder = CONVERTED / split / "labels"
    print(f"{split}: {folder.exists()}")

In [ ]:

from pathlib import Path

ROOT = Path("/content/drive/MyDrive/CNN/combined_vehicle_dataset")

print("Dataset exists:", ROOT.exists())
print("\nTop-level contents:")

if ROOT.exists():
    for item in sorted(ROOT.iterdir()):
        print("DIR " if item.is_dir() else "FILE", item.name)

    print("\nImage and label folders:")
    for item in ROOT.rglob("*"):
        if item.is_dir() and (
            "label" in item.name.lower()
            or "image" in item.name.lower()
        ):
            print(item.relative_to(ROOT))


In [ ]:
from pathlib import Path
from collections import Counter
import yaml

ROOT = Path("/content/drive/MyDrive/CNN/combined_vehicle_dataset")
LABEL_DIR = ROOT / "labels"

with open(ROOT / "data.yaml", "r") as f:
    data = yaml.safe_load(f)

names = data["names"]
if isinstance(names, list):
    names = dict(enumerate(names))
else:
    names = {int(k): v for k, v in names.items()}

counts = Counter()
label_files = list(LABEL_DIR.glob("*.txt"))
invalid_lines = []

for file in label_files:
    for line_no, line in enumerate(file.read_text().splitlines(), start=1):
        parts = line.split()
        if not parts:
            continue

        try:
            class_id = int(parts[0])
            if class_id not in names:
                invalid_lines.append((file.name, line_no, f"Unknown class ID: {class_id}"))
            else:
                counts[class_id] += 1
        except ValueError:
            invalid_lines.append((file.name, line_no, "Invalid class ID"))

print(f"Label files found: {len(label_files)}")
print("\nClass counts in converted dataset:")

for class_id, class_name in sorted(names.items()):
    print(f"{class_id}: {class_name} -> {counts[class_id]} objects")

print(f"\nInvalid class-ID lines: {len(invalid_lines)}")
for item in invalid_lines[:10]:
    print(item)

In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/CNN/combined_vehicle_dataset")

for folder_name in ["images", "labels"]:
    folder = ROOT / folder_name
    print(f"\n--- {folder_name.upper()} ---")
    print("Exists:", folder.exists())

    if folder.exists():
        files = [p for p in folder.rglob("*") if p.is_file()]
        print("Total files:", len(files))

        for p in files[:10]:
            print(p.relative_to(ROOT))

In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/CNN")

label_files = list(ROOT.rglob("*.txt"))

print("Total .txt files found:", len(label_files))

for p in label_files[:30]:
    print(p)

In [ ]:
from pathlib import Path
from collections import Counter
import yaml

ROOT = Path("/content/drive/MyDrive/CNN/combined_vehicle_dataset")
LABEL_DIR = ROOT / "labels"

with open(ROOT / "data.yaml", "r") as f:
    data = yaml.safe_load(f)

names = data["names"]
if isinstance(names, list):
    names = dict(enumerate(names))
else:
    names = {int(k): v for k, v in names.items()}

counts = Counter()
label_files = list(LABEL_DIR.rglob("*.txt"))

for file in label_files:
    for line in file.read_text().splitlines():
        parts = line.split()
        if parts:
            try:
                counts[int(parts[0])] += 1
            except ValueError:
                pass

print("Label files:", len(label_files))
print("\nObject counts:")
for class_id, name in sorted(names.items()):
    print(f"{class_id}: {name} -> {counts[class_id]}")

print("\nTarget classes:")
for class_id in [8, 9, 10]:
    print(f"{names[class_id]}: {counts[class_id]}")

In [ ]:
from pathlib import Path
from collections import Counter
import yaml

ROOT = Path("/content/drive/MyDrive/CNN/merged_vehicle_dataset")

with open(ROOT / "data.yaml", "r") as f:
    data = yaml.safe_load(f)

names = data["names"]
if isinstance(names, list):
    names = dict(enumerate(names))
else:
    names = {int(k): v for k, v in names.items()}

counts = Counter()
label_files = 0

for split in ["train", "valid", "test"]:
    label_dir = ROOT / split / "labels"

    if not label_dir.exists():
        print(f"Missing directory: {label_dir}")
        continue

    files = list(label_dir.rglob("*.txt"))
    label_files += len(files)

    for file in files:
        for line in file.read_text().splitlines():
            parts = line.split()
            if parts:
                try:
                    counts[int(parts[0])] += 1
                except ValueError:
                    pass

print("Merged label files:", label_files)
print("\nMerged object counts:")
for class_id, name in sorted(names.items()):
    print(f"{class_id}: {name} -> {counts[class_id]}")

print("\nTarget classes:")
for class_id, name in sorted(names.items()):
    if name in {"concrete_mixer", "tractor", "vehicle_truck"}:
        print(f"{name}: {counts[class_id]}")

Verify the converted labels and matching images

In [ ]:
from pathlib import Path
from collections import Counter
import yaml

ROOT = Path("/content/drive/MyDrive/CNN/combined_vehicle_dataset")

with open(ROOT / "data.yaml", "r") as f:
    data = yaml.safe_load(f)

names = data["names"]
if isinstance(names, list):
    names = dict(enumerate(names))
else:
    names = {int(k): v for k, v in names.items()}

target_names = {"concrete_mixer", "tractor", "vehicle_truck"}
target_ids = {i for i, name in names.items() if name in target_names}

print("Target class IDs:", target_ids)

for label_file in (ROOT / "labels").rglob("*.txt"):
    matching = []
    for line in label_file.read_text().splitlines():
        parts = line.split()
        if parts:
            try:
                class_id = int(parts[0])
                if class_id in target_ids:
                    matching.append((class_id, line))
            except ValueError:
                pass

    if matching:
        image_name = label_file.stem
        image_matches = [
            p for p in (ROOT / "images").rglob("*")
            if p.is_file() and p.stem == image_name
        ]
        print(f"\nLabel: {label_file}")
        print("Matching images:", [str(p) for p in image_matches])
        print("Target annotations:", matching)

Our final class mapping
We'll preserve your chosen classes and keep vehicle_truck unchanged.
ID	Unified class

0	bicycle

1	bus

2	car

3	bike

4	auto

5	truck

6	van

7	person

8	concrete_mixer

9	tractor

10	vehicle_truck

We'll exclude Cart.

One important detail: the traffic dataset contains a Suzuki class. We must inspect its examples before deciding whether it belongs to bike, car, or another category. We should not silently discard or mislabel those objects.

In [ ]:
from pathlib import Path
import yaml

BASE = Path("/content/drive/MyDrive/CNN")
SOURCE = BASE / "merged_vehicle_dataset_fixed"
OUTPUT = BASE / "vehicle_dataset_stratified_v2"

assert SOURCE.exists(), (
    f"Source not found: {SOURCE}\n"
    "Finish creating merged_vehicle_dataset_fixed first."
)

yaml_path = SOURCE / "data.yaml"
assert yaml_path.exists(), f"Missing YAML file: {yaml_path}"

with open(yaml_path, "r") as f:
    data = yaml.safe_load(f)

names = data["names"]
if isinstance(names, dict):
    names = [names[i] for i in sorted(names)]

print("Source:", SOURCE)
print("Output:", OUTPUT)
print("\nClass names:")
for i, name in enumerate(names):
    print(f"{i}: {name}")

for split in ["train", "valid", "test"]:
    images_dir = SOURCE / split / "images"
    labels_dir = SOURCE / split / "labels"

    if not images_dir.exists() or not labels_dir.exists():
        raise FileNotFoundError(
            f"Expected folders:\n{images_dir}\n{labels_dir}"
        )

    images = [
        p for p in images_dir.iterdir()
        if p.is_file() and p.suffix.lower() in
        {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    ]
    labels = list(labels_dir.glob("*.txt"))

    print(
        f"\n{split}: {len(images)} images, "
        f"{len(labels)} label files"
    )

required = {"concrete_mixer", "tractor", "vehicle_truck"}
missing = required - set(names)

if missing:
    raise ValueError(f"Missing expected classes: {missing}")

print("\nInitial folder check passed.")

Install the stratification package

In [ ]:
%pip -q install iterative-stratification

Validate labels and prepare class-presence data

In [ ]:
from pathlib import Path
from collections import Counter
import numpy as np

from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Collect only images with matching labels.
records = []
issues = []

for split in ["train", "valid", "test"]:
    images_dir = SOURCE / split / "images"
    labels_dir = SOURCE / split / "labels"

    for image_path in sorted(images_dir.iterdir()):
        if not image_path.is_file() or image_path.suffix.lower() not in IMAGE_EXTS:
            continue

        label_path = labels_dir / f"{image_path.stem}.txt"

        if not label_path.exists():
            issues.append(
                f"Missing label: {split}/{image_path.name}"
            )
            continue

        lines = label_path.read_text(
            encoding="utf-8", errors="replace"
        ).splitlines()

        present_classes = set()

        for line_no, line in enumerate(lines, start=1):
            if not line.strip():
                continue

            parts = line.split()

            if len(parts) != 5:
                issues.append(
                    f"Malformed line: {split}/{label_path.name}:{line_no}"
                )
                continue

            try:
                class_id = int(parts[0])
                coords = [float(x) for x in parts[1:]]
            except ValueError:
                issues.append(
                    f"Non-numeric annotation: "
                    f"{split}/{label_path.name}:{line_no}"
                )
                continue

            if not 0 <= class_id < len(names):
                issues.append(
                    f"Invalid class ID: {split}/{label_path.name}:{line_no}"
                )
                continue

            if not all(np.isfinite(x) and 0 <= x <= 1 for x in coords):
                issues.append(
                    f"Invalid bbox coordinates: "
                    f"{split}/{label_path.name}:{line_no}"
                )
                continue

            # Require positive width and height.
            if coords[2] <= 0 or coords[3] <= 0:
                issues.append(
                    f"Zero/negative bbox size: "
                    f"{split}/{label_path.name}:{line_no}"
                )
                continue

            present_classes.add(class_id)

        records.append({
            "image": image_path,
            "label": label_path,
            "classes": present_classes
        })

print("Matched image-label pairs:", len(records))
print("Validation issues:", len(issues))

if issues:
    print("\nFirst 30 issues:")
    for issue in issues[:30]:
        print("-", issue)

    raise ValueError(
        "\nFix the missing labels or invalid annotation lines before "
        "stratifying. No output dataset has been created by this cell."
    )

if not records:
    raise ValueError("No matched image-label pairs found.")

# Multi-label matrix: each row is an image; each column is a class.
Y = np.zeros((len(records), len(names)), dtype=int)

for i, record in enumerate(records):
    for class_id in record["classes"]:
        Y[i, class_id] = 1

print("\nImages containing each class:")
for class_id, name in enumerate(names):
    print(f"{name}: {int(Y[:, class_id].sum())}")

print("\nValidation passed. Ready to split.")

In [ ]:
import shutil
import yaml
import numpy as np
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

# Do not overwrite an existing output dataset.
if OUTPUT.exists():
    raise FileExistsError(
        f"Output already exists: {OUTPUT}\n"
        "Choose a new output folder before continuing."
    )

# Split 1: 70% train, 30% temporary.
split1 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_idx, temp_idx = next(
    split1.split(np.zeros(len(records)), Y)
)

# Split 2: temporary 30% -> 20% validation, 10% test.
split2 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=1/3,
    random_state=42
)

valid_rel_idx, test_rel_idx = next(
    split2.split(
        np.zeros(len(temp_idx)),
        Y[temp_idx]
    )
)

valid_idx = temp_idx[valid_rel_idx]
test_idx = temp_idx[test_rel_idx]

assignments = {
    "train": train_idx,
    "valid": valid_idx,
    "test": test_idx
}

# Create the output and copy image-label pairs.
for split, indices in assignments.items():
    images_out = OUTPUT / split / "images"
    labels_out = OUTPUT / split / "labels"

    images_out.mkdir(parents=True, exist_ok=True)
    labels_out.mkdir(parents=True, exist_ok=True)

    for idx in indices:
        record = records[int(idx)]

        shutil.copy2(
            record["image"],
            images_out / record["image"].name
        )
        shutil.copy2(
            record["label"],
            labels_out / record["label"].name
        )

# Write YOLO dataset configuration.
yaml_data = {
    "path": str(OUTPUT),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": len(names),
    "names": {i: name for i, name in enumerate(names)}
}

with open(OUTPUT / "data.yaml", "w") as f:
    yaml.safe_dump(yaml_data, f, sort_keys=False)

print("Stratified split created successfully!")
print("Output:", OUTPUT)

for split, indices in assignments.items():
    print(
        f"{split}: {len(indices)} images "
        f"({len(indices) / len(records) * 100:.2f}%)"
    )

In [ ]:
from pathlib import Path
from collections import Counter
import yaml

BASE = Path("/content/drive/MyDrive/CNN")

DATASETS = {
    "XML converted": BASE / "combined_vehicle_dataset",
    "Fixed merged": BASE / "merged_vehicle_dataset_fixed",
}

TARGET_CLASSES = {"concrete_mixer", "tractor", "vehicle_truck"}

for dataset_name, root in DATASETS.items():
    print(f"\n{'='*55}")
    print(dataset_name, "-", root)

    yaml_path = root / "data.yaml"
    if not yaml_path.exists():
        print("ERROR: data.yaml not found")
        continue

    with open(yaml_path, "r") as f:
        config = yaml.safe_load(f)

    class_names = config["names"]
    if isinstance(class_names, dict):
        class_names = [
            class_names[i] for i in sorted(class_names)
        ]

    target_ids = {
        i: name for i, name in enumerate(class_names)
        if name in TARGET_CLASSES
    }

    print("Target class IDs:", target_ids)
    counts = Counter()
    annotated_images = Counter()

    for label_path in (root / "labels").rglob("*.txt"):
        found_in_file = set()

        for line in label_path.read_text(
            encoding="utf-8", errors="replace"
        ).splitlines():
            parts = line.split()
            if not parts:
                continue

            try:
                class_id = int(parts[0])
            except ValueError:
                continue

            if class_id in target_ids:
                counts[class_id] += 1
                found_in_file.add(class_id)

        for class_id in found_in_file:
            annotated_images[class_id] += 1

    for class_id, name in target_ids.items():
        print(
            f"{name}: {counts[class_id]} objects "
            f"in {annotated_images[class_id]} label files"
        )

In [ ]:
from pathlib import Path

XML_DATA = Path(
    "/content/drive/MyDrive/CNN/combined_vehicle_dataset"
)

for folder in [
    XML_DATA / "images",
    XML_DATA / "labels",
]:
    print(f"\n{folder}")
    for split in ["train", "valid", "test"]:
        p = folder / split
        if p.exists():
            print(f"  {split}: {len(list(p.iterdir()))} files")
        else:
            print(f"  MISSING: {p}")

In [ ]:
from pathlib import Path
from collections import Counter
import shutil
import yaml

BASE = Path("/content/drive/MyDrive/CNN")

XML_DATA = BASE / "combined_vehicle_dataset"
MERGED = BASE / "merged_vehicle_dataset_fixed"
REPAIRED = BASE / "merged_vehicle_dataset_11class"

if REPAIRED.exists():
    raise FileExistsError(
        f"{REPAIRED} already exists. Choose another output folder."
    )

# Load class names from both datasets.
with open(MERGED / "data.yaml", "r") as f:
    merged_yaml = yaml.safe_load(f)

with open(XML_DATA / "data.yaml", "r") as f:
    xml_yaml = yaml.safe_load(f)

def ordered_names(config):
    names = config["names"]
    if isinstance(names, dict):
        return [names[i] for i in sorted(names)]
    return names

merged_names = ordered_names(merged_yaml)
xml_names = ordered_names(xml_yaml)

assert merged_names == xml_names, (
    "Class names or order differ. Do not merge until mappings are reconciled."
)

# Make a separate copy; the source remains untouched.
shutil.copytree(MERGED, REPAIRED)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
added = Counter()
skipped = []

for split in ["train", "valid", "test"]:
    src_images = XML_DATA / "images" / split
    src_labels = XML_DATA / "labels" / split
    dst_images = REPAIRED / split / "images"
    dst_labels = REPAIRED / split / "labels"

    for image_path in sorted(src_images.iterdir()):
        if not image_path.is_file() or image_path.suffix.lower() not in IMAGE_EXTS:
            continue

        label_path = src_labels / f"{image_path.stem}.txt"

        if not label_path.exists():
            skipped.append(f"{split}/{image_path.name}: missing label")
            continue

        # Avoid accidentally overwriting files already in the merged copy.
        if (
            (dst_images / image_path.name).exists()
            or (dst_labels / label_path.name).exists()
        ):
            skipped.append(f"{split}/{image_path.name}: filename collision")
            continue

        shutil.copy2(image_path, dst_images / image_path.name)
        shutil.copy2(label_path, dst_labels / label_path.name)
        added[split] += 1

# Save the repaired dataset YAML.
merged_yaml["path"] = str(REPAIRED)
merged_yaml["train"] = "train/images"
merged_yaml["val"] = "valid/images"
merged_yaml["test"] = "test/images"

with open(REPAIRED / "data.yaml", "w") as f:
    yaml.safe_dump(merged_yaml, f, sort_keys=False)

print("XML image-label pairs added:", dict(added))
print("Skipped:", len(skipped))
for item in skipped[:20]:
    print("-", item)

print("\nRepaired dataset:", REPAIRED)

In [ ]:
from pathlib import Path
from collections import Counter
import yaml

REPAIRED = Path(
    "/content/drive/MyDrive/CNN/merged_vehicle_dataset_11class"
)

with open(REPAIRED / "data.yaml", "r") as f:
    config = yaml.safe_load(f)

names = config["names"]
if isinstance(names, dict):
    names = [names[i] for i in sorted(names)]

object_counts = Counter()
images_per_class = Counter()
split_totals = {}

for split in ["train", "valid", "test"]:
    images_dir = REPAIRED / split / "images"
    labels_dir = REPAIRED / split / "labels"

    images = [
        p for p in images_dir.iterdir()
        if p.is_file() and p.suffix.lower()
        in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    ]

    split_totals[split] = len(images)

    for image_path in images:
        label_path = labels_dir / f"{image_path.stem}.txt"

        if not label_path.exists():
            print("MISSING LABEL:", split, image_path.name)
            continue

        classes_in_image = set()

        for line in label_path.read_text().splitlines():
            parts = line.split()
            if not parts:
                continue

            class_id = int(parts[0])
            if 0 <= class_id < len(names):
                object_counts[class_id] += 1
                classes_in_image.add(class_id)

        for class_id in classes_in_image:
            images_per_class[class_id] += 1

print("IMAGE COUNTS:", split_totals)
print("\nCLASS DISTRIBUTION")

for class_id, name in enumerate(names):
    print(
        f"{class_id:2}: {name:<18} "
        f"objects={object_counts[class_id]:<5} "
        f"images={images_per_class[class_id]}"
    )

In [ ]:
from pathlib import Path
from collections import Counter
import shutil
import yaml
import numpy as np

from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

# --------------------------------------------------
# 1. Paths
# --------------------------------------------------
BASE = Path("/content/drive/MyDrive/CNN")

SOURCE = BASE / "merged_vehicle_dataset_11class"
OUTPUT = BASE / "vehicle_dataset_stratified_70_20_10_final"

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

if not SOURCE.exists():
    raise FileNotFoundError(f"Source dataset not found: {SOURCE}")

if OUTPUT.exists():
    raise FileExistsError(
        f"Output already exists: {OUTPUT}\n"
        "Choose a new output folder to avoid overwriting."
    )

# --------------------------------------------------
# 2. Read class names
# --------------------------------------------------
with open(SOURCE / "data.yaml", "r") as f:
    config = yaml.safe_load(f)

names = config["names"]

if isinstance(names, dict):
    names = [names[i] for i in sorted(names)]

assert len(names) == 11, f"Expected 11 classes, found {len(names)}"

print("Classes:", names)

# --------------------------------------------------
# 3. Collect and validate image-label pairs
# --------------------------------------------------
records = []
issues = []

for source_split in ["train", "valid", "test"]:
    images_dir = SOURCE / source_split / "images"
    labels_dir = SOURCE / source_split / "labels"

    for image_path in sorted(images_dir.iterdir()):
        if not image_path.is_file() or image_path.suffix.lower() not in IMAGE_EXTS:
            continue

        label_path = labels_dir / f"{image_path.stem}.txt"

        if not label_path.exists():
            issues.append(f"Missing label: {image_path}")
            continue

        present_classes = set()

        for line_no, line in enumerate(
            label_path.read_text(
                encoding="utf-8", errors="replace"
            ).splitlines(),
            start=1
        ):
            if not line.strip():
                continue

            parts = line.split()

            if len(parts) != 5:
                issues.append(
                    f"Malformed annotation: {label_path}, line {line_no}"
                )
                continue

            try:
                class_id = int(parts[0])
                bbox = [float(v) for v in parts[1:]]
            except ValueError:
                issues.append(
                    f"Non-numeric annotation: {label_path}, line {line_no}"
                )
                continue

            if not 0 <= class_id < len(names):
                issues.append(
                    f"Invalid class ID: {label_path}, line {line_no}"
                )
                continue

            if not all(np.isfinite(v) and 0 <= v <= 1 for v in bbox):
                issues.append(
                    f"Invalid coordinates: {label_path}, line {line_no}"
                )
                continue

            if bbox[2] <= 0 or bbox[3] <= 0:
                issues.append(
                    f"Invalid bbox size: {label_path}, line {line_no}"
                )
                continue

            present_classes.add(class_id)

        records.append({
            "image": image_path,
            "label": label_path,
            "classes": present_classes
        })

if issues:
    print(f"Found {len(issues)} validation issues.")
    for issue in issues[:20]:
        print("-", issue)
    raise ValueError("Resolve annotation issues before splitting.")

if not records:
    raise ValueError("No image-label pairs found.")

# --------------------------------------------------
# 4. Build multilabel class-presence matrix
# --------------------------------------------------
Y = np.zeros((len(records), len(names)), dtype=int)

for i, record in enumerate(records):
    for class_id in record["classes"]:
        Y[i, class_id] = 1

print(f"\nValidated image-label pairs: {len(records)}")

print("\nClass presence before splitting:")
for class_id, name in enumerate(names):
    print(f"{name}: {int(Y[:, class_id].sum())} images")

# --------------------------------------------------
# 5. Split 70% train, 20% validation, 10% test
# --------------------------------------------------
split1 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_idx, temp_idx = next(
    split1.split(np.zeros(len(records)), Y)
)

split2 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=1/3,
    random_state=42
)

valid_rel_idx, test_rel_idx = next(
    split2.split(
        np.zeros(len(temp_idx)),
        Y[temp_idx]
    )
)

valid_idx = temp_idx[valid_rel_idx]
test_idx = temp_idx[test_rel_idx]

assignments = {
    "train": train_idx,
    "valid": valid_idx,
    "test": test_idx
}

# --------------------------------------------------
# 6. Copy image-label pairs to the new dataset
# --------------------------------------------------
for split, indices in assignments.items():
    images_out = OUTPUT / split / "images"
    labels_out = OUTPUT / split / "labels"

    images_out.mkdir(parents=True, exist_ok=True)
    labels_out.mkdir(parents=True, exist_ok=True)

    for idx in indices:
        record = records[int(idx)]

        shutil.copy2(
            record["image"],
            images_out / record["image"].name
        )

        shutil.copy2(
            record["label"],
            labels_out / record["label"].name
        )

# --------------------------------------------------
# 7. Write YOLO data.yaml
# --------------------------------------------------
yaml_data = {
    "path": str(OUTPUT),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": len(names),
    "names": {i: name for i, name in enumerate(names)}
}

with open(OUTPUT / "data.yaml", "w") as f:
    yaml.safe_dump(yaml_data, f, sort_keys=False)

# --------------------------------------------------
# 8. Report split sizes
# --------------------------------------------------
print("\nSTRATIFIED SPLIT COMPLETE")
print("Output:", OUTPUT)

for split, indices in assignments.items():
    print(
        f"{split}: {len(indices)} images "
        f"({len(indices) / len(records) * 100:.2f}%)"
    )

In [ ]:
from pathlib import Path

dataset = Path(
    "/content/drive/MyDrive/CNN/vehicle_dataset_stratified_70_20_10_final"
)

print("Dataset exists:", dataset.exists())

if dataset.exists():
    print("Contents:")
    for item in sorted(dataset.iterdir()):
        print(item.name)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

dataset = Path(
    "/content/drive/MyDrive/CNN/vehicle_dataset_stratified_70_20_10_final"
)

print("Dataset exists:", dataset.exists())
print("CNN folder exists:", dataset.parent.parent.exists())

if dataset.exists():
    print("\nDataset contents:")
    for item in sorted(dataset.iterdir()):
        print(item.name, "—", "Folder" if item.is_dir() else "File")

In [ ]:
!ls -lah "/content/drive/MyDrive/CNN/"
!ls -lah "/content/drive/MyDrive/CNN/vehicle_dataset_stratified_70_20_10_final/"

In [ ]:
from pathlib import Path

cnn = Path("/content/drive/MyDrive/CNN")

print("Final dataset exists:", (cnn / "vehicle_dataset_stratified_70_20_10_final").exists())
print("\nFolders containing 'stratified' or 'final':")

for p in sorted(cnn.iterdir()):
    if p.is_dir() and ("stratified" in p.name.lower() or "final" in p.name.lower()):
        print(p.name)

In [ ]:
from pathlib import Path

dataset = Path(
    "/content/drive/MyDrive/CNN/vehicle_dataset_stratified_70_20_10_final"
)

for split in ["train", "valid", "test"]:
    images = list((dataset / split / "images").glob("*"))
    labels = list((dataset / split / "labels").glob("*.txt"))
    print(f"{split}: {len(images)} images, {len(labels)} labels")

In [ ]:
from pathlib import Path

dataset = Path("/content/drive/MyDrive/CNN/vehicle_dataset_stratified_70_20_10_final")

print("Full path:", dataset.resolve())
print("Exists:", dataset.exists())
print("Is directory:", dataset.is_dir())
print("Contents:", [p.name for p in dataset.iterdir()] if dataset.exists() else "Not found")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!ls -lah "/content/drive/MyDrive/CNN/vehicle_dataset_stratified_70_20_10_final"

In [ ]:
from pathlib import Path
import shutil

# Existing dataset
source = Path(
    "/content/drive/MyDrive/CNN/vehicle_dataset_stratified_70_20_10_final"
)

# New location directly under My Drive
destination = Path(
    "/content/drive/MyDrive/YOLO11_Vehicle_Dataset"
)

# Verify source
if not source.exists():
    raise FileNotFoundError(f"Source dataset not found: {source}")

# Prevent accidental overwrite
if destination.exists():
    raise FileExistsError(
        f"Destination already exists: {destination}\n"
        "Choose another name to avoid overwriting files."
    )

print("Copying dataset to the new location...")
shutil.copytree(source, destination)

print("\nCopy completed!")
print("New location:", destination)
print("Dataset configuration exists:", (destination / "data.yaml").exists())

for split in ["train", "valid", "test"]:
    image_dir = destination / split / "images"
    label_dir = destination / split / "labels"

    images = sum(1 for p in image_dir.iterdir() if p.is_file())
    labels = sum(1 for p in label_dir.glob("*.txt") if p.is_file())

    print(f"{split}: {images} images, {labels} labels")

In [ ]:
from pathlib import Path

dataset = Path("/content/drive/MyDrive/YOLO11_Vehicle_Dataset")
yaml_file = dataset / "data.yaml"

print(yaml_file.read_text())

cofig


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

print("My Drive accessible:", os.path.exists("/content/drive/MyDrive"))

print("\nFolders inside MyDrive:")
for item in os.listdir("/content/drive/MyDrive"):
    print(item)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
My Drive accessible: True

Folders inside MyDrive:
Colab Notebooks (1)
POTHURAJUYASWANTH KUMAR-CCNAv7 Introduct-certificate.pdf
M3
Tutorial_EDIT.pdf
Classroom
SEM-5-20240321T171525Z-001.zip
Week-8.docx
IMG_20240930_162612_1.jpg
Colab Notebooks
Exam
ATS.docx
NDVTechsys_Solutions_Yaswanthkumar Pothuraju _JavaFullStack_2025.pdf
YaswanthKumar Pothuraju_javadeveloper.pdf
MVGR College of Engineering1-2-3.ecel
Cloud Computing (1).pdf
Cloud Computing.pdf
Leadership and Team Effectiveness (1).pdf
Leadership and Team Effectiveness.pdf
Resume.pdf
A3_CSD_Final_Project_PPT_Template.pptx
Screenrecording_20260813_003220.mp4
Audio 
Boom
pothurajuyaswanthkumar.pdf
CNN


In [ ]:
import os

search_root = "/content/drive/MyDrive"
target = "YOLO11_Vehicle_Dataset"

print("Searching Google Drive...")

for root, dirs, files in os.walk(search_root):
    if target in dirs:
        print("FOUND:", os.path.join(root, target))

Searching Google Drive...


KeyboardInterrupt: 

In [ ]:
from pathlib import Path

DATASET = Path("/content/drive/MyDrive/CNN/vehicle_dataset_stratified_70_20_10_final")
if not DATASET.exists():
    print("Dataset not found at the expected location.")
    print("Searching Google Drive for YOLO11_Vehicle_Dataset...")

    root = Path("/content/drive/MyDrive")
    matches = [
        p for p in root.rglob("YOLO11_Vehicle_Dataset")
        if p.is_dir()
    ]

    if matches:
        DATASET = matches[0]
        print("Dataset found at:", DATASET)
    else:
        raise FileNotFoundError(
            "YOLO11_Vehicle_Dataset was not found in MyDrive. "
            "Check the folder name and Drive location."
        )

print("\nUsing dataset:", DATASET)


Using dataset: /content/drive/MyDrive/CNN/vehicle_dataset_stratified_70_20_10_final


In [ ]:
from pathlib import Path
from collections import Counter
from PIL import Image
import numpy as np

CLASS_NAMES = [
    "bicycle", "bus", "car", "bike", "auto", "truck",
    "van", "person", "concrete_mixer", "tractor", "vehicle_truck"
]

SPLITS = ["train", "valid", "test"]
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

results = {}
total_images = 0

for split in SPLITS:
    image_dir = DATASET / split / "images"
    label_dir = DATASET / split / "labels"

    if not image_dir.is_dir() or not label_dir.is_dir():
        raise FileNotFoundError(
            f"Missing images or labels directory for split: {split}"
        )

    images = [
        p for p in image_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    ]

    image_map = {
        p.relative_to(image_dir).with_suffix("").as_posix(): p
        for p in images
    }

    label_map = {
        p.relative_to(label_dir).with_suffix("").as_posix(): p
        for p in label_dir.rglob("*.txt")
    }

    object_counts = Counter()
    image_counts = Counter()
    missing_labels = set(image_map) - set(label_map)
    orphan_labels = set(label_map) - set(image_map)

    for key in image_map.keys() & label_map.keys():
        seen_classes = set()

        for line in label_map[key].read_text().splitlines():
            values = line.split()

            if len(values) != 5:
                continue

            try:
                cls = int(values[0])
                coords = [float(v) for v in values[1:]]
            except ValueError:
                continue

            if 0 <= cls < len(CLASS_NAMES):
                object_counts[cls] += 1
                seen_classes.add(cls)

        for cls in seen_classes:
            image_counts[cls] += 1

    results[split] = {
        "images": len(images),
        "objects": object_counts,
        "class_images": image_counts,
        "missing": len(missing_labels),
        "orphan": len(orphan_labels),
    }

    total_images += len(images)

print("=" * 75)
print("DATASET SPLIT SUMMARY")
print("=" * 75)

for split in SPLITS:
    r = results[split]
    pct = 100 * r["images"] / total_images if total_images else 0

    print(
        f"{split:>5}: {r['images']:>5} images "
        f"({pct:5.2f}%) | "
        f"Missing labels: {r['missing']} | "
        f"Orphan labels: {r['orphan']}"
    )

print("\n" + "=" * 95)
print("CLASS DISTRIBUTION: PERCENTAGE OF IMAGES CONTAINING EACH CLASS")
print("=" * 95)

header = f"{'Class':<18}{'Train %':>12}{'Valid %':>12}{'Test %':>12}{'All %':>12}"
print(header)
print("-" * len(header))

for cls, name in enumerate(CLASS_NAMES):
    row = [name]

    for split in SPLITS:
        n = results[split]["images"]
        count = results[split]["class_images"][cls]
        pct = 100 * count / n if n else 0
        row.append(pct)

    all_count = sum(
        results[s]["class_images"][cls] for s in SPLITS
    )
    all_pct = 100 * all_count / total_images if total_images else 0

    print(
        f"{name:<18}"
        f"{row[1]:>11.2f}%"
        f"{row[2]:>11.2f}%"
        f"{row[3]:>11.2f}%"
        f"{all_pct:>11.2f}%"
    )

print("\n" + "=" * 75)
print("CLASS OBJECT COUNTS")
print("=" * 75)

print(f"{'Class':<18}{'Train':>10}{'Valid':>10}{'Test':>10}{'Total':>10}")

for cls, name in enumerate(CLASS_NAMES):
    counts = [results[s]["objects"][cls] for s in SPLITS]
    print(
        f"{name:<18}"
        f"{counts[0]:>10}"
        f"{counts[1]:>10}"
        f"{counts[2]:>10}"
        f"{sum(counts):>10}"
    )

FileNotFoundError: Missing images or labels directory for split: valid

In [ ]:
from pathlib import Path

DATASET = Path(
    "/content/drive/MyDrive/CNN/vehicle_dataset_stratified_70_20_10_final"
)

print("Dataset exists:", DATASET.exists())

if DATASET.exists():
    print("\nContents of dataset folder:")
    for item in sorted(DATASET.iterdir()):
        print("📁" if item.is_dir() else "📄", item.name)

    print("\nAll subdirectories (first 100):")
    for i, folder in enumerate(DATASET.rglob("*")):
        if folder.is_dir():
            print(folder.relative_to(DATASET))
            if i >= 99:
                break
else:
    print("Dataset path not found. Check that Google Drive is mounted.")

Dataset exists: True

Contents of dataset folder:
📁 train

All subdirectories (first 100):
train
train/images
train/labels


In [ ]:
%pip install -q iterative-stratification

In [ ]:
from pathlib import Path
from collections import Counter
from sklearn.model_selection import train_test_split
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
import shutil
import hashlib
import random
import numpy as np

# ============================================================
# CONFIGURATION
# ============================================================

SOURCE = Path(
    "/content/drive/MyDrive/CNN/merged_vehicle_dataset_11class"
)

OUTPUT = Path(
    "/content/drive/MyDrive/CNN/vehicle_dataset_stratified_70_20_10_final"
)

CLASS_NAMES = [
    "bicycle",
    "bus",
    "car",
    "bike",
    "auto",
    "truck",
    "van",
    "person",
    "concrete_mixer",
    "tractor",
    "vehicle_truck",
]

NUM_CLASSES = len(CLASS_NAMES)
IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"
}
SEED = 42

# ============================================================
# 1. VERIFY PATHS
# ============================================================

if not SOURCE.is_dir():
    raise FileNotFoundError(f"Source dataset not found: {SOURCE}")

if SOURCE.resolve() == OUTPUT.resolve():
    raise ValueError("Source and output paths must be different.")

print("Source:", SOURCE)
print("Output:", OUTPUT)

# ============================================================
# 2. COLLECT IMAGE-LABEL PAIRS FROM ALL SOURCE SPLITS
# ============================================================

pairs = []
missing_labels = []
seen_relative_paths = set()

for split in ["train", "valid", "test"]:
    image_dir = SOURCE / split / "images"
    label_dir = SOURCE / split / "labels"

    if not image_dir.is_dir() or not label_dir.is_dir():
        raise FileNotFoundError(
            f"Expected images and labels folders for source split: {split}"
        )

    for img in sorted(image_dir.rglob("*")):
        if not img.is_file() or img.suffix.lower() not in IMAGE_EXTENSIONS:
            continue

        relative_path = img.relative_to(image_dir)
        label = (label_dir / relative_path).with_suffix(".txt")

        if not label.is_file():
            missing_labels.append(str(img))
            continue

        # Check that the relative image path is unique across source splits.
        # Output filenames will be made unique separately.
        key = (split, str(relative_path))
        if key in seen_relative_paths:
            raise ValueError(f"Duplicate image path within split: {key}")
        seen_relative_paths.add(key)

        pairs.append({
            "image": img,
            "label": label,
            "source_split": split,
            "relative_path": relative_path,
        })

print("\nMatched image-label pairs:", len(pairs))
print("Images without label files:", len(missing_labels))

if missing_labels:
    print("\nExamples of images without labels:")
    for path in missing_labels[:10]:
        print(path)

    raise ValueError(
        "Some images have no corresponding label file. "
        "Resolve these before stratifying."
    )

if not pairs:
    raise ValueError("No image-label pairs found.")

# ============================================================
# 3. VALIDATE LABELS AND BUILD MULTILABEL MATRIX
# ============================================================

Y = np.zeros((len(pairs), NUM_CLASSES), dtype=int)
object_counts = Counter()
empty_label_files = 0
invalid_rows = []

for i, item in enumerate(pairs):
    label_path = item["label"]

    try:
        lines = label_path.read_text(encoding="utf-8").splitlines()
    except Exception as exc:
        raise ValueError(f"Cannot read label file {label_path}: {exc}")

    if not any(line.strip() for line in lines):
        empty_label_files += 1

    for line_no, line in enumerate(lines, start=1):
        line = line.strip()
        if not line:
            continue

        fields = line.split()

        if len(fields) != 5:
            invalid_rows.append(
                (str(label_path), line_no, "Expected 5 fields")
            )
            continue

        try:
            class_float = float(fields[0])
            coords = [float(v) for v in fields[1:]]
        except ValueError:
            invalid_rows.append(
                (str(label_path), line_no, "Non-numeric field")
            )
            continue

        class_id = int(class_float)

        if class_float != class_id or not (0 <= class_id < NUM_CLASSES):
            invalid_rows.append(
                (str(label_path), line_no, f"Invalid class ID: {fields[0]}")
            )
            continue

        if not np.isfinite(coords).all():
            invalid_rows.append(
                (str(label_path), line_no, "Non-finite coordinate")
            )
            continue

        # YOLO normalized x_center, y_center, width, height
        if any(v < 0 or v > 1 for v in coords):
            invalid_rows.append(
                (str(label_path), line_no, "Coordinate outside [0, 1]")
            )
            continue

        if coords[2] <= 0 or coords[3] <= 0:
            invalid_rows.append(
                (str(label_path), line_no, "Non-positive box width/height")
            )
            continue

        Y[i, class_id] = 1
        object_counts[class_id] += 1

if invalid_rows:
    print("\nInvalid annotation rows:", len(invalid_rows))
    for problem in invalid_rows[:20]:
        print(problem)

    raise ValueError(
        "Invalid annotations found. Fix these before creating the splits."
    )

print("Empty label files:", empty_label_files)
print("\nObject counts across all source images:")

for class_id, name in enumerate(CLASS_NAMES):
    print(
        f"{class_id:2} {name:16} "
        f"objects={object_counts[class_id]:5} "
        f"images={int(Y[:, class_id].sum()):5}"
    )

# Classes occurring in fewer than 3 images cannot reliably be represented
# in all three splits.
rare_classes = [
    CLASS_NAMES[c]
    for c in range(NUM_CLASSES)
    if Y[:, c].sum() < 3
]

if rare_classes:
    print(
        "\nWARNING: These classes occur in fewer than 3 images:",
        rare_classes
    )

# ============================================================
# 4. CHECK CONTENT DUPLICATES TO REDUCE DATA LEAKAGE RISK
# ============================================================

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

hash_to_indices = {}

for i, item in enumerate(pairs):
    digest = sha256_file(item["image"])
    hash_to_indices.setdefault(digest, []).append(i)

duplicate_groups = [
    indices for indices in hash_to_indices.values()
    if len(indices) > 1
]

if duplicate_groups:
    print(
        f"\nWARNING: Found {len(duplicate_groups)} groups of byte-identical "
        "images. These could leak across splits."
    )
    print(
        "Review duplicate groups before training. This stratification "
        "treats each file as a separate image."
    )
else:
    print("\nNo byte-identical image files detected.")

# ============================================================
# 5. MULTILABEL STRATIFICATION: 70% TRAIN, 30% REMAINDER
# ============================================================

indices = np.arange(len(pairs))

splitter = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=SEED
)

train_positions, remainder_positions = next(
    splitter.split(indices.reshape(-1, 1), Y)
)

# Split the 30% remainder into 20% validation and 10% test.
# 1/3 of the remainder goes to test.
remainder_Y = Y[remainder_positions]

splitter2 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=1/3,
    random_state=SEED
)

valid_relative, test_relative = next(
    splitter2.split(
        remainder_positions.reshape(-1, 1),
        remainder_Y
    )
)

valid_positions = remainder_positions[valid_relative]
test_positions = remainder_positions[test_relative]

splits = {
    "train": train_positions,
    "valid": valid_positions,
    "test": test_positions,
}

print("\nPlanned split sizes:")
for split_name, pos in splits.items():
    print(f"{split_name:5}: {len(pos):5} images ({len(pos)/len(pairs)*100:.2f}%)")

assert len(set(train_positions) & set(valid_positions)) == 0
assert len(set(train_positions) & set(test_positions)) == 0
assert len(set(valid_positions) & set(test_positions)) == 0
assert sum(len(pos) for pos in splits.values()) == len(pairs)

# ============================================================
# 6. PREPARE OUTPUT SAFELY
# ============================================================

# Do not overwrite a non-empty output directory automatically.
if OUTPUT.exists() and any(OUTPUT.iterdir()):
    raise FileExistsError(
        f"Output already exists and is not empty:\n{OUTPUT}\n\n"
        "Rename it or choose a new output path before continuing. "
        "Nothing has been overwritten."
    )

OUTPUT.mkdir(parents=True, exist_ok=True)

# ============================================================
# 7. COPY IMAGES AND LABELS
# ============================================================

for split_name, positions in splits.items():
    out_images = OUTPUT / split_name / "images"
    out_labels = OUTPUT / split_name / "labels"

    out_images.mkdir(parents=True, exist_ok=True)
    out_labels.mkdir(parents=True, exist_ok=True)

    for idx in positions:
        item = pairs[int(idx)]

        # Prefix with original split and use relative path to avoid collisions.
        relative = item["relative_path"]
        unique_stem = (
            f"{item['source_split']}__"
            + "__".join(relative.with_suffix("").parts)
        )

        image_dest = out_images / f"{unique_stem}{item['image'].suffix.lower()}"
        label_dest = out_labels / f"{unique_stem}.txt"

        shutil.copy2(item["image"], image_dest)
        shutil.copy2(item["label"], label_dest)

print("\nFiles copied successfully.")

# ============================================================
# 8. WRITE YOLO DATA YAML
# ============================================================

yaml_path = OUTPUT / "data.yaml"

yaml_text = (
    f"path: {OUTPUT}\n"
    "train: train/images\n"
    "val: valid/images\n"
    "test: test/images\n\n"
    f"nc: {NUM_CLASSES}\n"
    "names:\n"
    + "".join(f"  {i}: {name}\n" for i, name in enumerate(CLASS_NAMES))
)

yaml_path.write_text(yaml_text, encoding="utf-8")

print("\nCreated:", yaml_path)
print(yaml_text)

# ============================================================
# 9. AUDIT CLASS PRESENCE IN EACH SPLIT
# ============================================================

print("\nCLASS-PRESENCE AUDIT")
print("-" * 85)
print(f"{'Class':18} {'All':>7} {'Train':>7} {'Valid':>7} {'Test':>7}")

for class_id, class_name in enumerate(CLASS_NAMES):
    all_count = int(Y[:, class_id].sum())
    train_count = int(Y[train_positions, class_id].sum())
    valid_count = int(Y[valid_positions, class_id].sum())
    test_count = int(Y[test_positions, class_id].sum())

    print(
        f"{class_name:18} {all_count:7} "
        f"{train_count:7} {valid_count:7} {test_count:7}"
    )

print("\nFINAL DATASET")
for split_name in ["train", "valid", "test"]:
    img_count = sum(
        1 for p in (OUTPUT / split_name / "images").iterdir()
        if p.is_file()
    )
    lbl_count = sum(
        1 for p in (OUTPUT / split_name / "labels").iterdir()
        if p.is_file()
    )
    print(f"{split_name:5}: {img_count} images, {lbl_count} labels")

print("\nDataset output:", OUTPUT)
print("YAML file:", yaml_path)

Source: /content/drive/MyDrive/CNN/merged_vehicle_dataset_11class
Output: /content/drive/MyDrive/CNN/vehicle_dataset_stratified_70_20_10_final

Matched image-label pairs: 3069
Images without label files: 0
Empty label files: 599

Object counts across all source images:
 0 bicycle          objects=  193 images=  147
 1 bus              objects=  714 images=  475
 2 car              objects= 5862 images= 1982
 3 bike             objects= 1015 images=  482
 4 auto             objects=  779 images=  435
 5 truck            objects=  208 images=  134
 6 van              objects=  485 images=  268
 7 person           objects=  346 images=   33
 8 concrete_mixer   objects=   22 images=   22
 9 tractor          objects=   44 images=   34
10 vehicle_truck    objects=   70 images=   48

Review duplicate groups before training. This stratification treats each file as a separate image.

Planned split sizes:
train:  2148 images (69.99%)
valid:   614 images (20.01%)
test :   307 images (10.00%)


FileExistsError: Output already exists and is not empty:
/content/drive/MyDrive/CNN/vehicle_dataset_stratified_70_20_10_final

Rename it or choose a new output path before continuing. Nothing has been overwritten.

In [ ]:
from pathlib import Path

SOURCE = Path(
    "/content/drive/MyDrive/CNN/merged_vehicle_dataset_11class"
)

for split in ["train", "valid", "test"]:
    label_dir = SOURCE / split / "labels"

    empty_labels = []
    total_labels = 0

    for label_file in label_dir.rglob("*.txt"):
        total_labels += 1

        if not label_file.read_text(encoding="utf-8").strip():
            empty_labels.append(label_file)

    print(
        f"{split.upper():5} | "
        f"Total labels: {total_labels:4} | "
        f"Empty labels: {len(empty_labels):4}"
    )

TRAIN | Total labels: 1111 | Empty labels:  208


KeyboardInterrupt: 

In [ ]:
from pathlib import Path

SOURCE = Path(
    "/content/drive/MyDrive/CNN/merged_vehicle_dataset_11class"
)

OUTPUT = Path(
    "/content/drive/MyDrive/CNN/CNNDataset"
)

if not SOURCE.is_dir():
    raise FileNotFoundError(f"Source dataset not found: {SOURCE}")

if OUTPUT.exists() and any(OUTPUT.iterdir()):
    raise FileExistsError(
        f"Output folder already exists and is not empty: {OUTPUT}\n"
        "Choose another folder name to avoid overwriting files."
    )

OUTPUT.mkdir(parents=True, exist_ok=True)

print("✅ Source dataset:", SOURCE)
print("✅ New output path:", OUTPUT)
print("✅ Output folder is ready for stratification.")

✅ Source dataset: /content/drive/MyDrive/CNN/merged_vehicle_dataset_11class
✅ New output path: /content/drive/MyDrive/CNN/CNNDataset
✅ Output folder is ready for stratification.


In [ ]:
%pip install -q iterative-stratification

In [ ]:
from pathlib import Path
from collections import Counter
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
import numpy as np
import hashlib
import shutil

# =========================================================
# CONFIGURATION
# =========================================================

SOURCE = Path(
    "/content/drive/MyDrive/CNN/merged_vehicle_dataset_11class"
)

OUTPUT = Path(
    "/content/drive/MyDrive/CNN/CNNDataset"
)

CLASS_NAMES = [
    "bicycle",
    "bus",
    "car",
    "bike",
    "auto",
    "truck",
    "van",
    "person",
    "concrete_mixer",
    "tractor",
    "vehicle_truck",
]

NUM_CLASSES = len(CLASS_NAMES)
SEED = 42

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"
}

# =========================================================
# 1. CHECK SOURCE AND OUTPUT
# =========================================================

if not SOURCE.is_dir():
    raise FileNotFoundError(f"Source dataset not found: {SOURCE}")

if OUTPUT.exists() and any(OUTPUT.iterdir()):
    raise FileExistsError(
        f"Output folder is not empty: {OUTPUT}\n"
        "Choose a new output folder or empty it manually after backing "
        "up anything important."
    )

print("Source:", SOURCE)
print("Output:", OUTPUT)

# =========================================================
# 2. COLLECT IMAGE-LABEL PAIRS FROM ALL SOURCE SPLITS
# =========================================================

pairs = []
missing_labels = []

for source_split in ["train", "valid", "test"]:
    image_dir = SOURCE / source_split / "images"
    label_dir = SOURCE / source_split / "labels"

    if not image_dir.is_dir() or not label_dir.is_dir():
        raise FileNotFoundError(
            f"Missing images or labels directory for split: {source_split}"
        )

    for image_path in sorted(image_dir.rglob("*")):
        if (
            not image_path.is_file()
            or image_path.suffix.lower() not in IMAGE_EXTENSIONS
        ):
            continue

        relative_path = image_path.relative_to(image_dir)
        label_path = (label_dir / relative_path).with_suffix(".txt")

        if not label_path.is_file():
            missing_labels.append(str(image_path))
            continue

        pairs.append({
            "image": image_path,
            "label": label_path,
            "source_split": source_split,
            "relative_path": relative_path,
        })

print("\nMatched image-label pairs:", len(pairs))
print("Images missing label files:", len(missing_labels))

if missing_labels:
    print("\nExamples:")
    for p in missing_labels[:10]:
        print(p)

    raise ValueError(
        "Fix missing label files before stratification."
    )

if not pairs:
    raise ValueError("No image-label pairs found.")

# =========================================================
# 3. BUILD MULTILABEL MATRIX AND VALIDATE ANNOTATIONS
# =========================================================

Y = np.zeros((len(pairs), NUM_CLASSES), dtype=int)
object_counts = Counter()
empty_labels = 0
invalid_rows = []

for i, item in enumerate(pairs):
    label_path = item["label"]
    lines = label_path.read_text(encoding="utf-8").splitlines()

    if not any(line.strip() for line in lines):
        empty_labels += 1

    for line_no, line in enumerate(lines, start=1):
        line = line.strip()

        if not line:
            continue

        fields = line.split()

        if len(fields) != 5:
            invalid_rows.append(
                (str(label_path), line_no, "Expected 5 fields")
            )
            continue

        try:
            class_value = float(fields[0])
            coords = np.array(
                [float(v) for v in fields[1:]],
                dtype=float
            )
        except ValueError:
            invalid_rows.append(
                (str(label_path), line_no, "Non-numeric value")
            )
            continue

        class_id = int(class_value)

        if (
            class_value != class_id
            or not 0 <= class_id < NUM_CLASSES
        ):
            invalid_rows.append(
                (str(label_path), line_no, "Invalid class ID")
            )
            continue

        if not np.isfinite(coords).all():
            invalid_rows.append(
                (str(label_path), line_no, "Non-finite coordinates")
            )
            continue

        if np.any(coords < 0) or np.any(coords > 1):
            invalid_rows.append(
                (str(label_path), line_no, "Coordinates outside [0,1]")
            )
            continue

        if coords[2] <= 0 or coords[3] <= 0:
            invalid_rows.append(
                (str(label_path), line_no, "Invalid box width/height")
            )
            continue

        Y[i, class_id] = 1
        object_counts[class_id] += 1

if invalid_rows:
    print("\nInvalid annotation rows:", len(invalid_rows))

    for row in invalid_rows[:20]:
        print(row)

    raise ValueError("Fix invalid annotations before stratification.")

print("Empty label files:", empty_labels)

print("\nOverall class distribution:")
for class_id, class_name in enumerate(CLASS_NAMES):
    print(
        f"{class_id:2} {class_name:16} "
        f"objects={object_counts[class_id]:5} "
        f"images={Y[:, class_id].sum():5}"
    )

# =========================================================
# 4. GROUP BYTE-IDENTICAL IMAGES
# =========================================================

def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()

hash_groups = {}

for i, item in enumerate(pairs):
    digest = sha256_file(item["image"])
    hash_groups.setdefault(digest, []).append(i)

groups = list(hash_groups.values())

duplicate_groups = sum(1 for group in groups if len(group) > 1)

print("\nUnique image-content groups:", len(groups))
print("Groups containing duplicate files:", duplicate_groups)

# =========================================================
# 5. BUILD GROUP-LEVEL CLASS MATRIX
# =========================================================

# All duplicate files are assigned to the same split.
# A group's class vector is the union of classes in its images.
group_Y = np.zeros((len(groups), NUM_CLASSES), dtype=int)

for group_id, indices in enumerate(groups):
    group_Y[group_id] = np.max(Y[indices], axis=0)

if len(groups) < 3:
    raise ValueError("Too few unique image groups to make three splits.")

# =========================================================
# 6. STRATIFY GROUPS: 70% TRAIN, 30% REMAINDER
# =========================================================

all_group_ids = np.arange(len(groups))

splitter1 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=SEED
)

train_group_ids, remainder_group_ids = next(
    splitter1.split(
        all_group_ids.reshape(-1, 1),
        group_Y
    )
)

# Split remainder into 20% validation and 10% test overall.
remainder_Y = group_Y[remainder_group_ids]

splitter2 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=1/3,
    random_state=SEED
)

valid_rel, test_rel = next(
    splitter2.split(
        remainder_group_ids.reshape(-1, 1),
        remainder_Y
    )
)

valid_group_ids = remainder_group_ids[valid_rel]
test_group_ids = remainder_group_ids[test_rel]

split_groups = {
    "train": train_group_ids,
    "valid": valid_group_ids,
    "test": test_group_ids,
}

# Convert groups back to individual image indices.
split_indices = {}

for split_name, group_ids in split_groups.items():
    split_indices[split_name] = [
        image_idx
        for group_id in group_ids
        for image_idx in groups[int(group_id)]
    ]

# Verify there is no image-index overlap.
assert not (
    set(split_indices["train"]) & set(split_indices["valid"])
)
assert not (
    set(split_indices["train"]) & set(split_indices["test"])
)
assert not (
    set(split_indices["valid"]) & set(split_indices["test"])
)

print("\nPlanned split sizes:")
for split_name, indices in split_indices.items():
    print(
        f"{split_name:5}: {len(indices):5} images "
        f"({100 * len(indices) / len(pairs):.2f}%)"
    )

# =========================================================
# 7. CREATE OUTPUT DIRECTORIES
# =========================================================

OUTPUT.mkdir(parents=True, exist_ok=True)

for split_name in ["train", "valid", "test"]:
    (OUTPUT / split_name / "images").mkdir(
        parents=True, exist_ok=True
    )
    (OUTPUT / split_name / "labels").mkdir(
        parents=True, exist_ok=True
    )

# =========================================================
# 8. COPY IMAGES AND LABELS
# =========================================================

for split_name, indices in split_indices.items():
    for idx in indices:
        item = pairs[idx]

        # Prefix the original split to avoid filename collisions.
        relative_path = item["relative_path"]

        unique_stem = (
            f"{item['source_split']}__"
            + "__".join(relative_path.with_suffix("").parts)
        )

        image_dest = (
            OUTPUT / split_name / "images"
            / f"{unique_stem}{item['image'].suffix.lower()}"
        )

        label_dest = (
            OUTPUT / split_name / "labels"
            / f"{unique_stem}.txt"
        )

        shutil.copy2(item["image"], image_dest)
        shutil.copy2(item["label"], label_dest)

print("\nImages and labels copied successfully.")

# =========================================================
# 9. CREATE YOLO DATA YAML
# =========================================================

yaml_text = (
    f"path: {OUTPUT}\n"
    "train: train/images\n"
    "val: valid/images\n"
    "test: test/images\n\n"
    f"nc: {NUM_CLASSES}\n"
    "names:\n"
    + "".join(
        f"  {i}: {name}\n"
        for i, name in enumerate(CLASS_NAMES)
    )
)

(OUTPUT / "data.yaml").write_text(
    yaml_text,
    encoding="utf-8"
)

print("\nCreated data.yaml:")
print(yaml_text)

# =========================================================
# 10. FINAL CLASS-PRESENCE AUDIT
# =========================================================

print("\nCLASS-PRESENCE AUDIT")
print("-" * 78)
print(f"{'Class':18} {'All':>7} {'Train':>7} {'Valid':>7} {'Test':>7}")

for class_id, class_name in enumerate(CLASS_NAMES):
    all_count = int(Y[:, class_id].sum())

    train_count = int(
        Y[split_indices["train"], class_id].sum()
    )
    valid_count = int(
        Y[split_indices["valid"], class_id].sum()
    )
    test_count = int(
        Y[split_indices["test"], class_id].sum()
    )

    print(
        f"{class_name:18} {all_count:7} "
        f"{train_count:7} {valid_count:7} {test_count:7}"
    )

print("\nFINAL SPLIT COUNTS")

for split_name in ["train", "valid", "test"]:
    image_count = sum(
        1 for p in (OUTPUT / split_name / "images").iterdir()
        if p.is_file()
    )

    label_count = sum(
        1 for p in (OUTPUT / split_name / "labels").iterdir()
        if p.is_file()
    )

    print(
        f"{split_name:5}: "
        f"{image_count} images, {label_count} labels"
    )

print("\n✅ Stratified dataset created at:")
print(OUTPUT)
print("\nYOLO configuration:")
print(OUTPUT / "data.yaml")

Source: /content/drive/MyDrive/CNN/merged_vehicle_dataset_11class
Output: /content/drive/MyDrive/CNN/CNNDataset

Matched image-label pairs: 3069
Images missing label files: 0
Empty label files: 599

Overall class distribution:
 0 bicycle          objects=  193 images=  147
 1 bus              objects=  714 images=  475
 2 car              objects= 5862 images= 1982
 3 bike             objects= 1015 images=  482
 4 auto             objects=  779 images=  435
 5 truck            objects=  208 images=  134
 6 van              objects=  485 images=  268
 7 person           objects=  346 images=   33
 8 concrete_mixer   objects=   22 images=   22
 9 tractor          objects=   44 images=   34
10 vehicle_truck    objects=   70 images=   48

Unique image-content groups: 2969
Groups containing duplicate files: 100

Planned split sizes:
train:  2149 images (70.02%)
valid:   614 images (20.01%)
test :   306 images (9.97%)

Images and labels copied successfully.

Created data.yaml:
path: /content

In [ ]:
from pathlib import Path

DATASET = Path("/content/drive/MyDrive/CNN/CNNDataset")

for split in ["train", "valid", "test"]:
    images = DATASET / split / "images"
    labels = DATASET / split / "labels"

    image_count = (
        sum(1 for p in images.iterdir() if p.is_file())
        if images.exists() else 0
    )

    label_count = (
        sum(1 for p in labels.iterdir() if p.is_file())
        if labels.exists() else 0
    )

    print(f"{split}: {image_count} images, {label_count} labels")

train: 2149 images, 2149 labels
valid: 614 images, 614 labels
test: 306 images, 306 labels
